# Analisi, calcoli e visualizzazioni rispetto al campione di location di IMDb

File usati:

- geografie_cinema_ita_imdb.csv (dati con tutte le location)
- 1_reference_backup_film_IMDb.csv (dati con info generali singoli film)


# Primo step:

Caricare i file e leggerli

In [ ]:
import pandas as pd

file_path = '/content/geografie_cinema_ita_imdb.csv'
df = pd.read_csv(file_path)
display(df.head())

In [ ]:
import pandas as pd

file_path = '1_reference_backup_film_IMDb.csv'
df = pd.read_csv(file_path)
display(df.head())

# Step 2 Creazione di una mappa interattiva
Mappa semplice dove per ogni punto far comparire "cliccabile" il film di riferimento con "titolo", "anno" e link alla pagina imdb

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster

# Load the location data
df_geo = pd.read_csv('/content/geografie_cinema_ita_imdb.csv')

# Load the film info data
df_info = pd.read_csv('1_reference_backup_film_IMDb.csv')

# Merge the two dataframes to combine location and film information
# Assuming 'IMDb_ID' is the common key in both dataframes.
# We perform a left merge to keep all locations from df_geo and add film details from df_info.
df = pd.merge(df_geo, df_info[['IMDb_ID', 'URL']], on='IMDb_ID', how='left')

# Drop rows where 'Lat' or 'Long' are missing
df_locations = df.dropna(subset=['Lat', 'Long']).copy()

# Create a base map centered around the mean of latitudes and longitudes
# If no locations, center on Italy
if not df_locations.empty:
    map_center = [df_locations['Lat'].mean(), df_locations['Long'].mean()]
else:
    map_center = [41.902782, 12.496366] # Center of Italy (Rome)

m = folium.Map(location=map_center, zoom_start=6);

# Initialize MarkerCluster with maxClusterRadius
marker_cluster = MarkerCluster(maxClusterRadius=20).add_to(m)

# Add markers for each location to the MarkerCluster
for index, row in df_locations.iterrows():
    if pd.notnull(row['Lat']) and pd.notnull(row['Long']):
        # Create a hyperlink for the title using the 'URL' column from df_info (now 'URL_y')
        # Ensure 'URL_y', 'titolo', 'anno' are not NaN before creating the popup string.
        popup_text = f"<b>Titolo:</b> <a href=\"{row['URL_y']}\" target=\"_blank\">{row['titolo']}</a><br><b>Anno:</b> {row['anno']}"
        folium.Marker(
            location=[row['Lat'], row['Long']],
            popup=popup_text,
            tooltip=row['titolo'] # Show title on hover
        ).add_to(marker_cluster)

# Display the map
m

Step 2.5
Scaricare la mappa come file HTML navigabile

In [ ]:
map_file_path = 'film_locations_map.html'
m.save(map_file_path)
print(f"Map saved to {map_file_path}")

# Step 3. Calcolo il peso per continenti

Per calcolare il "peso" di ogni continente, attribuiamo all'associazione di un IMDb_ID al continente 1 punto indipendentemente da quante volte l'ID è associato allo stesso continente (es. un film ambientato in 5 location diverse d'Italia non deve valere come 5 film in Europa, ma solo 1). Mentre se lo stesso IMDb_ID è associato a più continenti, ognuno riceve un punto

In [ ]:
# Seleziona le colonne rilevanti
continent_data = df_geo[['IMDb_ID', 'continent']].copy()

# Rimuovi i NaN dalla colonna 'continent' prima di elaborare
continent_data.dropna(subset=['continent'], inplace=True)

# Rimuovi i duplicati per considerare ogni combinazione unica (IMDb_ID, continent) una sola volta
unique_continent_associations = continent_data.drop_duplicates(subset=['IMDb_ID', 'continent'])

# Calcola il 'peso' di ogni continente contando gli IMDb_ID unici associati
continent_weights = unique_continent_associations.groupby('continent')['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
continent_weights.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati
continent_weights = continent_weights.sort_values(by='Number of Unique IMDb_IDs', ascending=False)

display(continent_weights)

Scarico file.csv per visualizzazioni

In [ ]:
csv_file_path = '/content/continent_weights.csv'
continent_weights.to_csv(csv_file_path, index=False)
print(f"File CSV salvato in: {csv_file_path}")

Ne faccio una separazione che distingua per blocchi di 5 anni. I valori rappresentano il numero di "IMDb_ID" unici per ogni continente in quel quinquennio.

In [ ]:
import pandas as pd

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
continent_quinquennial_analysis_df = df_geo[['IMDb_ID', 'continent', 'anno']].copy()

# Rimuovi le righe con dati mancanti per 'continent' o 'anno'
continent_quinquennial_analysis_df.dropna(subset=['continent', 'anno'], inplace=True)

# Converti 'anno' a tipo numerico intero, gestendo eventuali errori di conversione
continent_quinquennial_analysis_df['anno'] = pd.to_numeric(continent_quinquennial_analysis_df['anno'], errors='coerce')
continent_quinquennial_analysis_df.dropna(subset=['anno'], inplace=True) # Rimuovi NaN introdotti dalla coercizione
continent_quinquennial_analysis_df['anno'] = continent_quinquennial_analysis_df['anno'].astype(int)

# Filtra i dati per l'intervallo di anni specificato (1945-2025)
continent_quinquennial_analysis_df = continent_quinquennial_analysis_df[
    (continent_quinquennial_analysis_df['anno'] >= 1945) &
    (continent_quinquennial_analysis_df['anno'] <= 2025)
]

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
continent_quinquennial_analysis_df['quinquennial'] = (continent_quinquennial_analysis_df['anno'] // 5) * 5

# Calcola i pesi: conta gli IMDb_ID unici per continente per ogni quinquennio
unique_associations_by_quinquennial_continent = continent_quinquennial_analysis_df.drop_duplicates(subset=['IMDb_ID', 'continent', 'quinquennial'])

# Poi, raggruppiamo per quinquennio e continente e contiamo gli IMDb_ID unici
continent_weights_by_quinquennial = unique_associations_by_quinquennial_continent.groupby(['quinquennial', 'continent'])['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
continent_weights_by_quinquennial.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati per quinquennio e poi per numero di IMDb_ID (discendente)
continent_weights_by_quinquennial.sort_values(by=['quinquennial', 'Number of Unique IMDb_IDs'], ascending=[True, False], inplace=True)

# Prepara i dati per la visualizzazione richiesta (continenti come righe, quinquenni come colonne)
continent_weights_pivot = continent_weights_by_quinquennial.pivot_table(
    index='continent',
    columns='quinquennial',
    values='Number of Unique IMDb_IDs'
).fillna(0)

# Assicurati che i quinquenni siano ordinati e che l'indice sia di tipo intero
continent_weights_pivot.columns = continent_weights_pivot.columns.astype(int)
continent_weights_pivot = continent_weights_pivot.sort_values(by='quinquennial', axis=1) # Ordina le colonne per quinquennio

# Rinomina le colonne per essere più descrittive, mostrando solo il range di anni
continent_weights_pivot.columns = [f"{col}-{col+4}" for col in continent_weights_pivot.columns]

display(continent_weights_pivot)

Scarico il file.csv

In [ ]:
csv_file_path_quinquennial = '/content/continent_weights_quinquennial.csv'
continent_weights_pivot.to_csv(csv_file_path_quinquennial, index=True)
print(f"File CSV salvato in: {csv_file_path_quinquennial}")

# Step 4 Calcolo il peso per country

Procedo ora con il calcolo del 'peso' per ogni paese, seguendo la stessa logica applicata in precedenza per i continenti.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Seleziona le colonne rilevanti
country_data = df_geo[['IMDb_ID', 'country']].copy()

# Rimuovi i NaN dalla colonna 'country' prima di elaborare
country_data.dropna(subset=['country'], inplace=True)

# Rimuovi i duplicati per considerare ogni combinazione unica (IMDb_ID, country) una sola volta
unique_country_associations = country_data.drop_duplicates(subset=['IMDb_ID', 'country'])

# Calcola il 'peso' di ogni country contando gli IMDb_ID unici associati
country_weights = unique_country_associations.groupby('country')['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
country_weights.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati
country_weights = country_weights.sort_values(by='Number of Unique IMDb_IDs', ascending=False)

display(country_weights.head(10))

Visualizzazione del peso dei Paesi

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x='Number of Unique IMDb_IDs', y='country', data=country_weights.head(20), palette='viridis') # Mostra i top 20 paesi
plt.title('Number of Unique IMDb_IDs per Country (Top 20)')
plt.xlabel('Number of Unique IMDb_IDs')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

Scarico file.csv

In [ ]:
csv_file_path_country = '/content/country_weights.csv'
country_weights.to_csv(csv_file_path_country, index=False)
print(f"File CSV salvato in: {csv_file_path_country}")

Ne faccio una separazione che distingua per blocchi di 5 anni. I valori rappresentano il numero di "IMDb_ID" unici per ogni paese in quel quinquennio.

In [ ]:
import pandas as pd

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
country_quinquennial_analysis_df = df_geo[['IMDb_ID', 'country', 'anno']].copy()

# Rimuovi le righe con dati mancanti per 'country' o 'anno'
country_quinquennial_analysis_df.dropna(subset=['country', 'anno'], inplace=True)

# Converti 'anno' a tipo numerico intero, gestendo eventuali errori di conversione
country_quinquennial_analysis_df['anno'] = pd.to_numeric(country_quinquennial_analysis_df['anno'], errors='coerce')
country_quinquennial_analysis_df.dropna(subset=['anno'], inplace=True) # Rimuovi NaN introdotti dalla coercizione
country_quinquennial_analysis_df['anno'] = country_quinquennial_analysis_df['anno'].astype(int)

# Filtra i dati per l'intervallo di anni specificato (1945-2025)
country_quinquennial_analysis_df = country_quinquennial_analysis_df[
    (country_quinquennial_analysis_df['anno'] >= 1945) &
    (country_quinquennial_analysis_df['anno'] <= 2025)
]

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
country_quinquennial_analysis_df['quinquennial'] = (country_quinquennial_analysis_df['anno'] // 5) * 5

# Calcola i pesi: conta gli IMDb_ID unici per paese per ogni quinquennio
unique_associations_by_quinquennial_country = country_quinquennial_analysis_df.drop_duplicates(subset=['IMDb_ID', 'country', 'quinquennial'])

# Poi, raggruppiamo per quinquennio e paese e contiamo gli IMDb_ID unici
country_weights_by_quinquennial = unique_associations_by_quinquennial_country.groupby(['quinquennial', 'country'])['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
country_weights_by_quinquennial.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati per quinquennio e poi per numero di IMDb_ID (discendente)
country_weights_by_quinquennial.sort_values(by=['quinquennial', 'Number of Unique IMDb_IDs'], ascending=[True, False], inplace=True)

# Prepara i dati per la visualizzazione richiesta (paesi come righe, quinquenni come colonne)
country_weights_pivot = country_weights_by_quinquennial.pivot_table(
    index='country',
    columns='quinquennial',
    values='Number of Unique IMDb_IDs'
).fillna(0)

# Assicurati che i quinquenni siano ordinati e che l'indice sia di tipo intero
country_weights_pivot.columns = country_weights_pivot.columns.astype(int)
country_weights_pivot = country_weights_pivot.sort_values(by='quinquennial', axis=1) # Ordina le colonne per quinquennio

# Rinomina le colonne per essere più descrittive, mostrando solo il range di anni
country_weights_pivot.columns = [f"{col}-{col+4}" for col in country_weights_pivot.columns]

display(country_weights_pivot.head(10)) # Mostra le prime 10 righe per visione d'insieme

Scarico file.csv

In [ ]:
csv_file_path_country_quinquennial = '/content/country_weights_quinquennial.csv'
country_weights_pivot.to_csv(csv_file_path_country_quinquennial, index=True)
print(f"File CSV salvato in: {csv_file_path_country_quinquennial}")

# Step 5 Calcolo il peso per regione

Procedo ora con il calcolo del 'peso' per ogni regione, seguendo la stessa logica applicata in precedenza per i continenti.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Seleziona le colonne rilevanti
region_data = df_geo[['IMDb_ID', 'region']].copy()

# Rimuovi i NaN dalla colonna 'region' prima di elaborare
region_data.dropna(subset=['region'], inplace=True)

# Rimuovi i duplicati per considerare ogni combinazione unica (IMDb_ID, region) una sola volta
unique_region_associations = region_data.drop_duplicates(subset=['IMDb_ID', 'region'])

# Calcola il 'peso' di ogni region contando gli IMDb_ID unici associati
region_weights = unique_region_associations.groupby('region')['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
region_weights.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati
region_weights = region_weights.sort_values(by='Number of Unique IMDb_IDs', ascending=False)

display(region_weights.head(10))

Visualizzazione del peso delle Regioni (comunque da rifare)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x='Number of Unique IMDb_IDs', y='region', data=region_weights.head(20), palette='viridis') # Mostra i top 20 regioni
plt.title('Number of Unique IMDb_IDs per Region (Top 20)')
plt.xlabel('Number of Unique IMDb_IDs')
plt.ylabel('Region')
plt.tight_layout()
plt.show()

Scarico file.csv

In [ ]:
csv_file_path_region = '/content/region_weights.csv'
region_weights.to_csv(csv_file_path_region, index=False)
print(f"File CSV salvato in: {csv_file_path_region}")

Ne faccio una separazione che distingua per blocchi di 5 anni. I valori rappresentano il numero di "IMDb_ID" unici per ogni regione in quel quinquennio.

In [ ]:
import pandas as pd

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
region_quinquennial_analysis_df = df_geo[['IMDb_ID', 'region', 'anno']].copy()

# Rimuovi le righe con dati mancanti per 'region' o 'anno'
region_quinquennial_analysis_df.dropna(subset=['region', 'anno'], inplace=True)

# Converti 'anno' a tipo numerico intero, gestendo eventuali errori di conversione
region_quinquennial_analysis_df['anno'] = pd.to_numeric(region_quinquennial_analysis_df['anno'], errors='coerce')
region_quinquennial_analysis_df.dropna(subset=['anno'], inplace=True) # Rimuovi NaN introdotti dalla coercizione
region_quinquennial_analysis_df['anno'] = region_quinquennial_analysis_df['anno'].astype(int)

# Filtra i dati per l'intervallo di anni specificato (1945-2025)
region_quinquennial_analysis_df = region_quinquennial_analysis_df[
    (region_quinquennial_analysis_df['anno'] >= 1945) &
    (region_quinquennial_analysis_df['anno'] <= 2025)
]

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
region_quinquennial_analysis_df['quinquennial'] = (region_quinquennial_analysis_df['anno'] // 5) * 5

# Calcola i pesi: conta gli IMDb_ID unici per regione per ogni quinquennio
unique_associations_by_quinquennial_region = region_quinquennial_analysis_df.drop_duplicates(subset=['IMDb_ID', 'region', 'quinquennial'])

# Poi, raggruppiamo per quinquennio e regione e contiamo gli IMDb_ID unici
region_weights_by_quinquennial = unique_associations_by_quinquennial_region.groupby(['quinquennial', 'region'])['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
region_weights_by_quinquennial.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati per quinquennio e poi per numero di IMDb_ID (discendente)
region_weights_by_quinquennial.sort_values(by=['quinquennial', 'Number of Unique IMDb_IDs'], ascending=[True, False], inplace=True)

# Prepara i dati per la visualizzazione richiesta (regioni come righe, quinquenni come colonne)
region_weights_pivot = region_weights_by_quinquennial.pivot_table(
    index='region',
    columns='quinquennial',
    values='Number of Unique IMDb_IDs'
).fillna(0)

# Assicurati che i quinquenni siano ordinati e che l'indice sia di tipo intero
region_weights_pivot.columns = region_weights_pivot.columns.astype(int)
region_weights_pivot = region_weights_pivot.sort_values(by='quinquennial', axis=1) # Ordina le colonne per quinquennio

# Rinomina le colonne per essere più descrittive, mostrando solo il range di anni
region_weights_pivot.columns = [f"{col}-{col+4}" for col in region_weights_pivot.columns]

display(region_weights_pivot.head(10)) # Mostra le prime 10 righe per visione d'insieme

In [ ]:
csv_file_path_region_quinquennial = '/content/region_weights_quinquennial.csv'
region_weights_pivot.to_csv(csv_file_path_region_quinquennial, index=True)
print(f"File CSV salvato in: {csv_file_path_region_quinquennial}")

## Frequenza relativa per regione e quinquennio

In [ ]:
import pandas as pd

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
region_quinquennial_analysis_df = df_geo[['IMDb_ID', 'region', 'anno']].copy()

# Rimuovi le righe con dati mancanti per 'region' o 'anno'
region_quinquennial_analysis_df.dropna(subset=['region', 'anno'], inplace=True)

# Converti 'anno' a tipo numerico intero, gestendo eventuali errori di conversione
region_quinquennial_analysis_df['anno'] = pd.to_numeric(region_quinquennial_analysis_df['anno'], errors='coerce')
region_quinquennial_analysis_df.dropna(subset=['anno'], inplace=True) # Rimuovi NaN introdotti dalla coercizione
region_quinquennial_analysis_df['anno'] = region_quinquennial_analysis_df['anno'].astype(int)

# Filtra i dati per l'intervallo di anni specificato (1945-2025)
region_quinquennial_analysis_df = region_quinquennial_analysis_df[
    (region_quinquennial_analysis_df['anno'] >= 1945) &
    (region_quinquennial_analysis_df['anno'] <= 2025)
]

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
region_quinquennial_analysis_df['quinquennial'] = (region_quinquennial_analysis_df['anno'] // 5) * 5

# Calcola i pesi: conta gli IMDb_ID unici per regione per ogni quinquennio
unique_associations_by_quinquennial_region = region_quinquennial_analysis_df.drop_duplicates(subset=['IMDb_ID', 'region', 'quinquennial'])

# Poi, raggruppiamo per quinquennio e regione e contiamo gli IMDb_ID unici
region_weights_by_quinquennial = unique_associations_by_quinquennial_region.groupby(['quinquennial', 'region'])['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
region_weights_by_quinquennial.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati per quinquennio e poi per numero di IMDb_ID (discendente)
region_weights_by_quinquennial.sort_values(by=['quinquennial', 'Number of Unique IMDb_IDs'], ascending=[True, False], inplace=True)

# Calcola il totale di IMDb_ID unici per ogni quinquennio
total_imdb_ids_per_quinquennial_region = region_weights_by_quinquennial.groupby('quinquennial')['Number of Unique IMDb_IDs'].sum().reset_index()
total_imdb_ids_per_quinquennial_region.rename(columns={'Number of Unique IMDb_IDs': 'Total_IMDb_IDs_in_quinquennial'}, inplace=True)

# Unisci il totale al DataFrame dei pesi regionali per quinquennio
region_relative_frequency_df = pd.merge(
    region_weights_by_quinquennial,
    total_imdb_ids_per_quinquennial_region,
    on='quinquennial',
    how='left'
)

# Calcola la frequenza relativa
region_relative_frequency_df['Relative_Frequency'] = (
    region_relative_frequency_df['Number of Unique IMDb_IDs'] /
    region_relative_frequency_df['Total_IMDb_IDs_in_quinquennial']
)

# Prepara i dati per la visualizzazione pivot (regioni come righe, quinquenni come colonne)
region_relative_frequency_pivot = region_relative_frequency_df.pivot_table(
    index='region',
    columns='quinquennial',
    values='Relative_Frequency'
).fillna(0)

# Assicurati che i quinquenni siano ordinati e che l'indice sia di tipo intero
region_relative_frequency_pivot.columns = region_relative_frequency_pivot.columns.astype(int)
region_relative_frequency_pivot = region_relative_frequency_pivot.sort_values(by='quinquennial', axis=1) # Ordina le colonne per quinquennio

# Rinomina le colonne per essere più descrittive, mostrando solo il range di anni
region_relative_frequency_pivot.columns = [f"{col}-{col+4}" for col in region_relative_frequency_pivot.columns]

display(region_relative_frequency_pivot.head(10)) # Mostra le prime 10 righe per visione d'insieme

Scarico il file .csv con le frequenze relative per regione e quinquennio.

In [ ]:
csv_file_path_region_relative_quinquennial = '/content/region_relative_frequency_quinquennial.csv'
region_relative_frequency_pivot.to_csv(csv_file_path_region_relative_quinquennial, index=True)
print(f"File CSV salvato in: {csv_file_path_region_relative_quinquennial}")

Frequenza relativa periodo (2000-2025)

In [ ]:
import pandas as pd

# Filter region_relative_frequency_df for quinquennial periods from 2000 to 2025
# Assuming quinquennial periods are represented by their start year (e.g., 2000 for 2000-2004, 2020 for 2020-2024)
filtered_quinquennial_df = region_relative_frequency_df[
    (region_relative_frequency_df['quinquennial'] >= 2000) &
    (region_relative_frequency_df['quinquennial'] <= 2020) # Max quinquennial start year for the 2000-2025 range
].copy()

# Calculate total unique IMDb_IDs per region for the 2000-2025 period
total_unique_imdb_ids_per_region_2000_2025 = filtered_quinquennial_df.groupby('region')['Number of Unique IMDb_IDs'].sum().reset_index()

# Calculate the overall total unique IMDb_IDs for the 2000-2025 period across all regions
overall_total_imdb_ids_2000_2025 = filtered_quinquennial_df['Number of Unique IMDb_IDs'].sum()

# Calculate the relative frequency for each region for the 2000-2025 period
region_relative_frequency_2000_2025_df = total_unique_imdb_ids_per_region_2000_2025.copy()
region_relative_frequency_2000_2025_df['Relative_IMDb_Frequency_2000_2025'] = (
    region_relative_frequency_2000_2025_df['Number of Unique IMDb_IDs'] /
    overall_total_imdb_ids_2000_2025
)

# Keep only the 'region' and the newly calculated 'Relative_IMDb_Frequency_2000_2025' columns
region_relative_frequency_2000_2025_df = region_relative_frequency_2000_2025_df[['region', 'Relative_IMDb_Frequency_2000_2025']]

display(region_relative_frequency_2000_2025_df)

In [ ]:
csv_file_path_region_relative_frequency_2000_2025 = '/content/region_relative_frequency_2000_2025.csv'
region_relative_frequency_2000_2025_df.to_csv(csv_file_path_region_relative_frequency_2000_2025, index=True)
print(f"File CSV salvato in: {csv_file_path_region_relative_frequency_2000_2025}")

Frequenza relativa periodo 1945-2025

In [ ]:
import pandas as pd

# Ensure IMDb_ID is string type for consistent merges
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)

# Create a DataFrame with relevant columns from df_geo and filter years
region_full_analysis_df = df_geo[['IMDb_ID', 'region', 'anno']].copy()
region_full_analysis_df.dropna(subset=['region', 'anno'], inplace=True)
region_full_analysis_df['anno'] = pd.to_numeric(region_full_analysis_df['anno'], errors='coerce')
region_full_analysis_df.dropna(subset=['anno'], inplace=True)
region_full_analysis_df['anno'] = region_full_analysis_df['anno'].astype(int)

# Filter for the specified year range (1945-2025)
region_full_analysis_df = region_full_analysis_df[
    (region_full_analysis_df['anno'] >= 1945) &
    (region_full_analysis_df['anno'] <= 2025)
]

# Remove duplicate (IMDb_ID, region) pairs to get unique associations
unique_associations_full = region_full_analysis_df.drop_duplicates(subset=['IMDb_ID', 'region'])

# Group by region and count the unique IMDb_IDs
region_counts_full = unique_associations_full.groupby('region')['IMDb_ID'].nunique().reset_index()
region_counts_full.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Calculate the total number of unique IMDb_IDs nationally
total_imdb_ids_national_full = region_counts_full['Number of Unique IMDb_IDs'].sum()

# Calculate the relative frequency for each region
region_relative_frequency_full_data = region_counts_full.copy()
region_relative_frequency_full_data['Relative_Frequency'] = (
    region_relative_frequency_full_data['Number of Unique IMDb_IDs'] /
    total_imdb_ids_national_full
)

print("Frequenza Relativa per Regione (1945-2025):")
display(region_relative_frequency_full_data)

# Save the DataFrame to a CSV file
csv_file_path_region_relative_frequency_1945_2025 = '/content/region_relative_frequency_1945_2025.csv'
region_relative_frequency_full_data.to_csv(csv_file_path_region_relative_frequency_1945_2025, index=False)
print(f"File CSV salvato in: {csv_file_path_region_relative_frequency_1945_2025}")

# Coefficente di variazione regionale per quinquenni.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming region_relative_frequency_pivot is available from previous steps
# (df_region_relative_frequency_pivot from cell 03516493)

# 1. Calculate the mean and standard deviation of relative frequency for each region
# The pivot table has regions as index and quinquennial periods as columns
regional_stats = region_relative_frequency_pivot.agg(['mean', 'std'], axis=1).reset_index()
regional_stats.rename(columns={'index': 'region'}, inplace=True)

# 2. Calculate the Coefficient of Variation (CV)
# CV = standard deviation / mean
regional_stats['Coefficiente_di_Variazione'] = regional_stats['std'] / regional_stats['mean']

# Handle cases where mean is zero to avoid division by zero errors
# If mean is 0, CV is 0 (no variation)
regional_stats['Coefficiente_di_Variazione'].fillna(0, inplace=True)
regional_stats.loc[regional_stats['mean'] == 0, 'Coefficiente_di_Variazione'] = 0

# Sort the results by CV in descending order
regional_cv = regional_stats.sort_values(by='Coefficiente_di_Variazione', ascending=False)

display(regional_cv.head())

# Save regional_cv to CSV
regional_cv_path = '/content/regional_cv.csv'
regional_cv.to_csv(regional_cv_path, index=False)
print(f"File CSV salvato in: {regional_cv_path}")

# 3. Visualize the Coefficient of Variation for each region
plt.figure(figsize=(14, 8))
sns.barplot(
    x='Coefficiente_di_Variazione',
    y='region',
    data=regional_cv.head(20), # Show all regions or top 20 for clarity
    palette='viridis'
)
plt.title('Coefficiente di Variazione della Frequenza Relativa IMDb per Regione (1945-2025)')
plt.xlabel('Coefficiente di Variazione')
plt.ylabel('Regione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 4. Temporal trend of the Coefficient of Variation (CV)
# Calculate CV for each quinquennial period across all regions

# Transpose the pivot table to have quinquennial periods as index and regions as columns
quinquennial_regional_data = region_relative_frequency_pivot.T

# Calculate mean and standard deviation across regions for each quinquennial period
quinquennial_cv_data = quinquennial_regional_data.agg(['mean', 'std'], axis=1).reset_index()
quinquennial_cv_data.rename(columns={'index': 'quinquennial'}, inplace=True)

# Convert quinquennial column to numeric for plotting
quinquennial_cv_data['quinquennial_start_year'] = quinquennial_cv_data['quinquennial'].apply(lambda x: int(x.split('-')[0]))

# Calculate CV
quinquennial_cv_data['Coefficiente_di_Variazione_Temporale'] = quinquennial_cv_data['std'] / quinquennial_cv_data['mean']

# Handle cases where mean is zero
quinquennial_cv_data['Coefficiente_di_Variazione_Temporale'].fillna(0, inplace=True)
quinquennial_cv_data.loc[quinquennial_cv_data['mean'] == 0, 'Coefficiente_di_Variazione_Temporale'] = 0

display(quinquennial_cv_data.head())

# Save quinquennial_cv_data to CSV
quinquennial_cv_data_path = '/content/quinquennial_cv_data.csv'
quinquennial_cv_data.to_csv(quinquennial_cv_data_path, index=False)
print(f"File CSV salvato in: {quinquennial_cv_data_path}")

# 5. Visualize the temporal trend of CV
plt.figure(figsize=(14, 7))
sns.lineplot(
    x='quinquennial_start_year',
    y='Coefficiente_di_Variazione_Temporale',
    data=quinquennial_cv_data,
    marker='o'
)
plt.title('Andamento Temporale del Coefficiente di Variazione (1945-2025) tra le Regioni')
plt.xlabel('Quinquennio (anno di inizio)')
plt.ylabel('Coefficiente di Variazione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Coefficiente di Variazione (CV) su base Annuale

Questa sezione ricalcola e visualizza il Coefficiente di Variazione (CV) sia a livello regionale che temporale, utilizzando la frequenza relativa dei film su base **annuale** invece che quinquennale. Questo permette un'analisi più dettagliata delle fluttuazioni.

**1. Calcolo della Frequenza Relativa Annuale per Regione**

Prima di calcolare il CV, è necessario ottenere la frequenza relativa di film per ogni regione per ogni anno. Questo significa che per ogni anno e per ogni regione, determiniamo la proporzione di film unici associati a quella regione rispetto al totale dei film unici a livello nazionale in quello stesso anno.

In [ ]:
import pandas as pd

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
region_annual_analysis_df = df_geo[['IMDb_ID', 'region', 'anno']].copy()

# Rimuovi le righe con dati mancanti per 'region' o 'anno'
region_annual_analysis_df.dropna(subset=['region', 'anno'], inplace=True)

# Converti 'anno' a tipo numerico intero, gestendo eventuali errori di conversione
region_annual_analysis_df['anno'] = pd.to_numeric(region_annual_analysis_df['anno'], errors='coerce')
region_annual_analysis_df.dropna(subset=['anno'], inplace=True) # Rimuovi NaN introdotti dalla coercizione
region_annual_analysis_df['anno'] = region_annual_analysis_df['anno'].astype(int)

# Filtra i dati per l'intervallo di anni specificato (o usa min/max da df_geo)
# Manteniamo l'intervallo 1945-2025 per coerenza con le analisi precedenti
region_annual_analysis_df = region_annual_analysis_df[
    (region_annual_analysis_df['anno'] >= 1945) &
    (region_annual_analysis_df['anno'] <= 2025)
]

# Calcola i pesi: conta gli IMDb_ID unici per regione per ogni anno
unique_associations_by_annual_region = region_annual_analysis_df.drop_duplicates(subset=['IMDb_ID', 'region', 'anno'])

# Poi, raggruppiamo per anno e regione e contiamo gli IMDb_ID unici
region_weights_by_annual = unique_associations_by_annual_region.groupby(['anno', 'region'])['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
region_weights_by_annual.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Ordina i risultati per anno e poi per numero di IMDb_ID (discendente)
region_weights_by_annual.sort_values(by=['anno', 'Number of Unique IMDb_IDs'], ascending=[True, False], inplace=True)

# Calcola il totale di IMDb_ID unici per ogni anno
total_imdb_ids_per_annual_region = region_weights_by_annual.groupby('anno')['Number of Unique IMDb_IDs'].sum().reset_index()
total_imdb_ids_per_annual_region.rename(columns={'Number of Unique IMDb_IDs': 'Total_IMDb_IDs_in_annual'}, inplace=True)

# Unisci il totale al DataFrame dei pesi regionali per anno
region_relative_frequency_annual_df = pd.merge(
    region_weights_by_annual,
    total_imdb_ids_per_annual_region,
    on='anno',
    how='left'
)

# Calcola la frequenza relativa
region_relative_frequency_annual_df['Relative_Frequency'] = (
    region_relative_frequency_annual_df['Number of Unique IMDb_IDs'] /
    region_relative_frequency_annual_df['Total_IMDb_IDs_in_annual']
)

# Prepara i dati per la visualizzazione pivot (regioni come righe, anni come colonne)
region_relative_frequency_annual_pivot = region_relative_frequency_annual_df.pivot_table(
    index='region',
    columns='anno',
    values='Relative_Frequency'
).fillna(0)

# Assicurati che gli anni siano ordinati e che l'indice sia di tipo intero
region_relative_frequency_annual_pivot.columns = region_relative_frequency_annual_pivot.columns.astype(int)
region_relative_frequency_annual_pivot = region_relative_frequency_annual_pivot.sort_values(by='anno', axis=1) # Ordina le colonne per anno

display(region_relative_frequency_annual_pivot.head(10)) # Mostra le prime 10 righe per visione d'insieme

## Calcolo e Visualizzazione del Coefficiente di Variazione (CV) su base Annuale

Con i dati di frequenza relativa annuale per regione, possiamo ora calcolare il CV per analizzare la variabilità nel tempo e tra le regioni.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Regional CV Calculation (Annual Data) ---
# 1. Calculate the mean and standard deviation of relative frequency for each region
# The pivot table has regions as index and annual periods as columns
regional_stats_annual = region_relative_frequency_annual_pivot.agg(['mean', 'std'], axis=1).reset_index()
regional_stats_annual.rename(columns={'index': 'region'}, inplace=True)

# 2. Calculate the Coefficient of Variation (CV)
# CV = standard deviation / mean
regional_stats_annual['Coefficiente_di_Variazione'] = regional_stats_annual['std'] / regional_stats_annual['mean']

# Handle cases where mean is zero to avoid division by zero errors
# If mean is 0, CV is 0 (no variation)
regional_stats_annual['Coefficiente_di_Variazione'].fillna(0, inplace=True)
regional_stats_annual.loc[regional_stats_annual['mean'] == 0, 'Coefficiente_di_Variazione'] = 0

# Sort the results by CV in descending order
regional_cv_annual = regional_stats_annual.sort_values(by='Coefficiente_di_Variazione', ascending=False)

print("Regional Coefficient of Variation (Annual Data):")
display(regional_cv_annual.head())

# 3. Visualize the Coefficient of Variation for each region
plt.figure(figsize=(14, 8))
sns.barplot(
    x='Coefficiente_di_Variazione',
    y='region',
    data=regional_cv_annual.head(20), # Mostra i top 20 regioni
    palette='viridis'
)
plt.title('Coefficiente di Variazione della Frequenza Relativa IMDb per Regione (Annuale)')
plt.xlabel('Coefficiente di Variazione')
plt.ylabel('Regione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Temporal CV Visualization (Quinquennial Data) ---
# The quinquennial_cv_data is pre-calculated in cell 8FEckmjRwR5o.
# 4. Display the temporal Coefficient of Variation on a quinquennial basis
print("\nTemporal Coefficient of Variation (Quinquennial Data):")
display(quinquennial_cv_data.head())

# 5. Visualize the temporal trend of CV on a quinquennial basis
plt.figure(figsize=(14, 7))
sns.lineplot(
    x='quinquennial_start_year', # Use quinquennial_start_year from quinquennial_cv_data
    y='Coefficiente_di_Variazione_Temporale',
    data=quinquennial_cv_data, # Use quinquennial_cv_data for plotting
    marker='o'
)
plt.title('Andamento Temporale del Coefficiente di Variazione (Quinquennale) tra le Regioni')
plt.xlabel('Quinquennio (anno di inizio)')
plt.ylabel('Coefficiente di Variazione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Salvo csv con valori unici per regione sia 1945-2025, sia 2000-2025

In [ ]:
# Save regional_cv_annual to CSV
csv_file_path_regional_cv_annual = '/content/regional_cv_annual.csv'
regional_cv_annual.to_csv(csv_file_path_regional_cv_annual, index=False)
print(f"File CSV salvato in: {csv_file_path_regional_cv_annual}")

# Filter region_relative_frequency_annual_pivot for the 2000-2025 period
# Assuming region_relative_frequency_annual_pivot is available from previous cells
# Get columns that represent years between 2000 and 2025
year_columns_2000_2025 = [col for col in region_relative_frequency_annual_pivot.columns if col >= 2000 and col <= 2025]

# Create a new pivot table with only these years
regional_relative_frequency_annual_pivot_2000_2025 = region_relative_frequency_annual_pivot[year_columns_2000_2025]

# Calculate mean and standard deviation for the 2000-2025 period for each region
regional_stats_annual_2000_2025 = regional_relative_frequency_annual_pivot_2000_2025.agg(['mean', 'std'], axis=1).reset_index()
regional_stats_annual_2000_2025.rename(columns={'index': 'region'}, inplace=True)

# Calculate the Coefficient of Variation (CV) for 2000-2025
regional_stats_annual_2000_2025['Coefficiente_di_Variazione'] = regional_stats_annual_2000_2025['std'] / regional_stats_annual_2000_2025['mean']

# Handle cases where mean is zero to avoid division by zero errors
regional_stats_annual_2000_2025['Coefficiente_di_Variazione'].fillna(0, inplace=True)
regional_stats_annual_2000_2025.loc[regional_stats_annual_2000_2025['mean'] == 0, 'Coefficiente_di_Variazione'] = 0

# Sort the results by CV in descending order
regional_cv_annual_2000_2025 = regional_stats_annual_2000_2025.sort_values(by='Coefficiente_di_Variazione', ascending=False)

print("\nRegional Coefficient of Variation (2000-2025 Annual Data):")
display(regional_cv_annual_2000_2025.head())

# Save regional_cv_annual_2000_2025 to CSV
csv_file_path_regional_cv_annual_2000_2025 = '/content/regional_cv_annual_2000_2025.csv'
regional_cv_annual_2000_2025.to_csv(csv_file_path_regional_cv_annual_2000_2025, index=False)
print(f"File CSV salvato in: {csv_file_path_regional_cv_annual_2000_2025}")

Csv salvato con dati temporali raggruppati quinquennialmente

In [ ]:
# Save quinquennial_cv_data to CSV
csv_file_path_quinquennial_temporal_cv_data = '/content/quinquennial_temporal_cv_data.csv'
quinquennial_cv_data.to_csv(csv_file_path_quinquennial_temporal_cv_data, index=False)
print(f"File CSV salvato in: {csv_file_path_quinquennial_temporal_cv_data}")

# PARTE TRE
Mobilità

In [ ]:
import pandas as pd

# 1. Seleziona le colonne rilevanti da df_geo
# `df_geo` contiene già 'IMDb_ID', 'region' e 'titolo'
region_variety_data = df_geo[['IMDb_ID', 'region', 'titolo']].copy()

# 2. Rimuovi le righe con valori NaN nella colonna 'region'
region_variety_data.dropna(subset=['region'], inplace=True)

# 3. Rimuovi i duplicati per contare ogni combinazione unica (IMDb_ID, region) una sola volta
unique_imdb_region_pairs = region_variety_data.drop_duplicates(subset=['IMDb_ID', 'region'])

# 4. Calcola il numero di regioni uniche per ogni IMDb_ID
region_variety_counts = unique_imdb_region_pairs.groupby('IMDb_ID')['region'].nunique().reset_index()
region_variety_counts.rename(columns={'region': 'Region_Variety'}, inplace=True)

# 5. Trova la massima varietà di regioni
max_region_variety = region_variety_counts['Region_Variety'].max()
print(f"La massima varietà di regioni per un singolo film è: {max_region_variety}")

# 6. Filtra per ottenere gli IMDb_ID con la massima varietà
films_with_max_variety_ids = region_variety_counts[region_variety_counts['Region_Variety'] == max_region_variety]

# 7. Unisci con df_info per ottenere i titoli, assicurandoti di ottenere titoli unici
# df_info è più affidabile per i titoli unici associati a IMDb_ID
final_result = pd.merge(
    films_with_max_variety_ids,
    df_info[['IMDb_ID', 'titolo']].drop_duplicates(), # Drop duplicates from df_info just in case
    on='IMDb_ID',
    how='left'
)

display(final_result)

# Se vuoi anche visualizzare tutte le regioni associate a questi film, possiamo farlo in un passo successivo


A partire da questo valore. Assumento 5 come valore di massima varietà regionale (valore che possiamo dare per rappresentativo); misuriamo questo come massimo valore di "mobilità" e operazionalizziamo un ***“coefficiente di mobilità regionale”*** in cui a ogni film (ovvero ogni singolo IMDb_ID) viene attribuito 0 punti se alla loro voce nel dataset corrisponde una sola regione (anche se più volte, ma una unica) e 1 se alla loro voce nel dataset corrisponde il più alto numero di co-occorrenze regionali, ovvero 5.

In [ ]:
# Calcola il "coefficiente di mobilità regionale"
# La formula normalizza il valore di 'Region_Variety' tra 0 e 1, dove 1 regione = 0 e max_region_variety (5) regioni = 1.
# Questo permette di avere un valore proporzionale per le varietà intermedie, come suggerito dall'operazionalizzazione di un "coefficiente".

# Assicurati che max_region_variety sia un valore valido e maggiore di 1 per evitare divisioni per zero.
if max_region_variety > 1:
    region_variety_counts['Mobility_Coefficient'] = (
        region_variety_counts['Region_Variety'] - 1
    ) / (max_region_variety - 1)
else:
    # Se tutti i film sono girati in una sola regione (max_region_variety = 1),
    # il coefficiente di mobilità è 0 per tutti.
    region_variety_counts['Mobility_Coefficient'] = 0

# Visualizza i risultati con il nuovo coefficiente
display(region_variety_counts.head())

# Puoi anche visualizzare i film con il coefficiente più alto (o più basso) per un controllo
print("\nFilm con il coefficiente di mobilità più alto (coefficiente = 1):")
display(region_variety_counts[region_variety_counts['Mobility_Coefficient'] == 1].merge(df_info[['IMDb_ID', 'titolo']].drop_duplicates(), on='IMDb_ID', how='left'))

print("\nFilm con il coefficiente di mobilità più basso (coefficiente = 0):")
display(region_variety_counts[region_variety_counts['Mobility_Coefficient'] == 0].merge(df_info[['IMDb_ID', 'titolo']].drop_duplicates(), on='IMDb_ID', how='left').head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Import numpy for arange

plt.figure(figsize=(10, 6))

# Define custom bin edges to capture the discrete values properly
# The bins should be centered around 0, 0.25, 0.5, 0.75, 1.0
# For example, a bin for 0.00 would be from -0.125 to 0.125
# A bin for 0.25 would be from 0.125 to 0.375
bin_edges = np.arange(-0.125, 1.126, 0.25) # Start slightly before 0, end slightly after 1, step by 0.25

sns.histplot(region_variety_counts['Mobility_Coefficient'], bins=bin_edges, kde=False, stat='count') # Use stat='count' for frequency on y-axis, kde=False to simplify
plt.title('Distribuzione del Coefficiente di Mobilità Regionale')
plt.xlabel('Regional mobility score')
plt.ylabel('Number of films')
plt.xticks(np.arange(0, 1.01, 0.25)) # Set x-ticks to correspond to the discrete values
plt.grid(True, linestyle='--', alpha=0.7)

# Calcola la media del Coefficiente di Mobilità
mean_mobility_coefficient = region_variety_counts['Mobility_Coefficient'].mean()

# Aggiungi una linea verticale rossa che indica la media
plt.axvline(mean_mobility_coefficient, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_mobility_coefficient:.2f}')
plt.legend()

plt.show()

# Save the region_variety_counts DataFrame to CSV
csv_file_path_mobility_coefficient = '/content/mobility_coefficient.csv'
region_variety_counts.to_csv(csv_file_path_mobility_coefficient, index=False)
print(f"File CSV salvato in: {csv_file_path_mobility_coefficient}")

Csv

In [ ]:
import pandas as pd

# Group by 'Mobility_Coefficient' and count the number of unique IMDb_IDs for each coefficient
mobility_summary = region_variety_counts.groupby('Mobility_Coefficient')['IMDb_ID'].nunique().reset_index()
mobility_summary.rename(columns={'IMDb_ID': 'Number of Films'}, inplace=True)

display(mobility_summary)

# Save the summary to a CSV file
csv_file_path_mobility_summary = '/content/mobility_coefficient_summary.csv'
mobility_summary.to_csv(csv_file_path_mobility_summary, index=False)
print(f"File CSV riassuntivo salvato in: {csv_file_path_mobility_summary}")

Calcolo coefficente mobilità (2000-2025)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure IMDb_ID is of the same type for consistent merging if not already
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)

# 1. Filter df_geo for the period 2000-2025
# Create a temporary DataFrame with relevant columns for year filtering and analysis
region_variety_data_2000_2025 = df_geo[['IMDb_ID', 'region', 'anno']].copy()

# Drop rows with missing 'region' or 'anno'
region_variety_data_2000_2025.dropna(subset=['region', 'anno'], inplace=True)

# Convert 'anno' to numeric, coercing errors, then drop NaNs from 'anno'
region_variety_data_2000_2025['anno'] = pd.to_numeric(region_variety_data_2000_2025['anno'], errors='coerce')
region_variety_data_2000_2025.dropna(subset=['anno'], inplace=True)
region_variety_data_2000_2025['anno'] = region_variety_data_2000_2025['anno'].astype(int)

# Filter for the specified year range
region_variety_data_2000_2025 = region_variety_data_2000_2025[
    (region_variety_data_2000_2025['anno'] >= 2000) &
    (region_variety_data_2000_2025['anno'] <= 2025)
].copy()

# 2. Calculate the number of unique regions for each IMDb_ID within this period
# Drop duplicates to count each unique (IMDb_ID, region) association only once
unique_imdb_region_pairs_2000_2025 = region_variety_data_2000_2025.drop_duplicates(subset=['IMDb_ID', 'region'])

# Count unique regions per IMDb_ID
region_variety_counts_2000_2025 = unique_imdb_region_pairs_2000_2025.groupby('IMDb_ID')['region'].nunique().reset_index()
region_variety_counts_2000_2025.rename(columns={'region': 'Region_Variety'}, inplace=True)

# 3. Find the maximum variety of regions in this period
max_region_variety_2000_2025 = region_variety_counts_2000_2025['Region_Variety'].max()

print(f"La massima varietà di regioni per un singolo film nel periodo 2000-2025 è: {max_region_variety_2000_2025}")

# 4. Calculate the Mobility_Coefficient
if max_region_variety_2000_2025 > 1:
    region_variety_counts_2000_2025['Mobility_Coefficient'] = (
        region_variety_counts_2000_2025['Region_Variety'] - 1
    ) / (max_region_variety_2000_2025 - 1)
else:
    region_variety_counts_2000_2025['Mobility_Coefficient'] = 0

# Display a sample of the results
display(region_variety_counts_2000_2025.head())

# 5. Visualize the distribution of the Mobility_Coefficient
plt.figure(figsize=(10, 6))

# Define custom bin edges for histogram
bin_edges = np.arange(-0.125, 1.126, 0.25)

sns.histplot(region_variety_counts_2000_2025['Mobility_Coefficient'], bins=bin_edges, kde=False, stat='count')
plt.title('Distribuzione del Coefficiente di Mobilità Regionale (2000-2025)')
plt.xlabel('Coefficiente di Mobilità')
plt.ylabel('Frequenza (Numero di Film)')
plt.xticks(np.arange(0, 1.01, 0.25))
plt.grid(True, linestyle='--', alpha=0.7)

# Calculate and add the mean mobility coefficient
mean_mobility_coefficient_2000_2025 = region_variety_counts_2000_2025['Mobility_Coefficient'].mean()
plt.axvline(
    mean_mobility_coefficient_2000_2025,
    color='red',
    linestyle='dashed',
    linewidth=2,
    label=f'Media: {mean_mobility_coefficient_2000_2025:.2f}'
)
plt.legend()
plt.show()

# 6. Save the full results to a CSV file
csv_file_path_mobility_coefficient_2000_2025 = '/content/mobility_coefficient_2000_2025.csv'
region_variety_counts_2000_2025.to_csv(csv_file_path_mobility_coefficient_2000_2025, index=False)
print(f"File CSV salvato in: {csv_file_path_mobility_coefficient_2000_2025}")

# 7. Create and save a summary CSV
mobility_summary_2000_2025 = region_variety_counts_2000_2025.groupby('Mobility_Coefficient')['IMDb_ID'].nunique().reset_index()
mobility_summary_2000_2025.rename(columns={'IMDb_ID': 'Number of Films'}, inplace=True)

print("\nSummary of Mobility Coefficients (2000-2025):")
display(mobility_summary_2000_2025)

csv_file_path_mobility_summary_2000_2025 = '/content/mobility_coefficient_summary_2000_2025.csv'
mobility_summary_2000_2025.to_csv(csv_file_path_mobility_summary_2000_2025, index=False)
print(f"File CSV riassuntivo salvato in: {csv_file_path_mobility_summary_2000_2025}")

Calcolo coefficente mobilità senza Lazio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Filter df_geo to exclude 'Lazio' from the 'region' column
df_geo_no_lazio = df_geo[df_geo['region'] != 'Lazio'].copy()

# 2. Recalculate region_variety_data for films NOT involving Lazio
# Ensure IMDb_ID is string type for consistent merging if not already
df_geo_no_lazio['IMDb_ID'] = df_geo_no_lazio['IMDb_ID'].astype(str)

# Select relevant columns and remove NaN regions
region_variety_data_no_lazio = df_geo_no_lazio[['IMDb_ID', 'region', 'titolo']].copy()
region_variety_data_no_lazio.dropna(subset=['region'], inplace=True)

# Remove duplicates to count each unique (IMDb_ID, region) association only once
unique_imdb_region_pairs_no_lazio = region_variety_data_no_lazio.drop_duplicates(subset=['IMDb_ID', 'region'])

# Calculate the number of regions each IMDb_ID appears in
region_variety_counts_no_lazio = unique_imdb_region_pairs_no_lazio.groupby('IMDb_ID')['region'].nunique().reset_index()
region_variety_counts_no_lazio.rename(columns={'region': 'Region_Variety'}, inplace=True)

# 3. Recalculate max_region_variety from the new region_variety_counts_no_lazio
max_region_variety_no_lazio = region_variety_counts_no_lazio['Region_Variety'].max()

print(f"La massima varietà di regioni per un singolo film (escludendo Lazio) è: {max_region_variety_no_lazio}")

# 4. Calculate the Mobility_Coefficient using the new max_region_variety
if max_region_variety_no_lazio > 1:
    region_variety_counts_no_lazio['Mobility_Coefficient'] = (
        region_variety_counts_no_lazio['Region_Variety'] - 1
    ) / (max_region_variety_no_lazio - 1)
else:
    region_variety_counts_no_lazio['Mobility_Coefficient'] = 0

display(region_variety_counts_no_lazio.head())

# 5. Generate a histogram for the new Mobility_Coefficient
plt.figure(figsize=(10, 6))

# Define custom bin edges similar to the previous mobility coefficient plot
bin_edges_no_lazio = np.arange(-0.125, 1.126, 0.25)

sns.histplot(
    region_variety_counts_no_lazio['Mobility_Coefficient'],
    bins=bin_edges_no_lazio,
    kde=False,
    stat='count'
)
plt.title('Distribuzione del Coefficiente di Mobilità Regionale (Escludendo Lazio)')
plt.xlabel('Coefficiente di Mobilità')
plt.ylabel('Frequenza (Numero di Film)')
plt.xticks(np.arange(0, 1.01, 0.25)) # Set x-ticks to correspond to the discrete values
plt.grid(True, linestyle='--', alpha=0.7)

# Calcola la media del Coefficiente di Mobilità
mean_mobility_coefficient_no_lazio = region_variety_counts_no_lazio['Mobility_Coefficient'].mean()

# Aggiungi una linea verticale rossa che indica la media
plt.axvline(
    mean_mobility_coefficient_no_lazio,
    color='red',
    linestyle='dashed',
    linewidth=2,
    label=f'Media: {mean_mobility_coefficient_no_lazio:.2f}'
)
plt.legend()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assumiamo che region_variety_counts e df_geo siano disponibili dalle celle precedenti.
# region_variety_counts contiene: IMDb_ID, Region_Variety, Mobility_Coefficient
# df_geo contiene: IMDb_ID, region

# Assicuriamo che le colonne 'IMDb_ID' siano di tipo stringa per un merge consistente
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)
region_variety_counts['IMDb_ID'] = region_variety_counts['IMDb_ID'].astype(str)

# Step 1: Identifichiamo i film con Mobility_Coefficient == 0 (cioè, Region_Variety == 1)
films_mobility_0 = region_variety_counts[region_variety_counts['Mobility_Coefficient'] == 0].copy()

# Step 2: Uniamo questi film con df_geo per ottenere la regione associata.
# Per i film con Region_Variety = 1, dovrebbe esserci solo una regione unica per IMDb_ID.
# Filtriamo df_geo per includere solo gli IMDb_ID presenti in films_mobility_0.
films_mobility_0_regions = df_geo[df_geo['IMDb_ID'].isin(films_mobility_0['IMDb_ID'])].copy()

# Rimuoviamo i duplicati per assicurare che ogni IMDb_ID abbia una singola regione associata
films_mobility_0_regions_unique = films_mobility_0_regions.drop_duplicates(subset=['IMDb_ID', 'region'])

# Uniamo le informazioni sulla regione al dataframe films_mobility_0
films_mobility_0_with_region = pd.merge(
    films_mobility_0,
    films_mobility_0_regions_unique[['IMDb_ID', 'region']],
    on='IMDb_ID',
    how='left'
)

# Step 3: Identifichiamo i film 'solo Lazio'
lazio_only_films_ids = films_mobility_0_with_region[
    films_mobility_0_with_region['region'] == 'Lazio'
]['IMDb_ID'].unique()

# Step 4: Escludiamo i film 'solo Lazio' dal dataframe originale region_variety_counts per la visualizzazione
mobility_coefficient_filtered = region_variety_counts[
    ~region_variety_counts['IMDb_ID'].isin(lazio_only_films_ids)
].copy()

# Stampiamo alcune statistiche per verifica
print(f"Totale film prima del filtraggio: {len(region_variety_counts)}")
print(f"Film con Mobility_Coefficient == 0: {len(films_mobility_0)}")
print(f"Film identificati come 'solo Lazio': {len(lazio_only_films_ids)}")
print(f"Totale film dopo il filtraggio (escludendo 'solo Lazio'): {len(mobility_coefficient_filtered)}")

# Step 5: Visualizziamo la distribuzione del Coefficiente di Mobilità per i dati filtrati
plt.figure(figsize=(10, 6))

# Definiamo i bordi personalizzati per i bin per catturare correttamente i valori discreti
bin_edges = np.arange(-0.125, 1.126, 0.25)

sns.histplot(
    mobility_coefficient_filtered['Mobility_Coefficient'],
    bins=bin_edges,
    kde=False,
    stat='count'
)
plt.title('Distribuzione del Coefficiente di Mobilità Regionale (Escludendo film solo Lazio)')
plt.xlabel('Coefficiente di Mobilità')
plt.ylabel('Frequenza (Numero di Film)')
plt.xticks(np.arange(0, 1.01, 0.25)) # Impostiamo gli x-ticks per corrispondere ai valori discreti
plt.grid(True, linestyle='--', alpha=0.7)

# Calcoliamo e aggiungiamo la media
mean_mobility_coefficient_filtered = mobility_coefficient_filtered['Mobility_Coefficient'].mean()
plt.axvline(
    mean_mobility_coefficient_filtered,
    color='red',
    linestyle='dashed',
    linewidth=2,
    label=f'Media: {mean_mobility_coefficient_filtered:.2f}'
)
plt.legend()
plt.show()

# Opzionale: Salviamo i dati filtrati del coefficiente di mobilità
csv_file_path_mobility_coefficient_no_lazio_only = '/content/mobility_coefficient_no_lazio_only.csv'
mobility_coefficient_filtered.to_csv(csv_file_path_mobility_coefficient_no_lazio_only, index=False)
print(f"File CSV salvato in: {csv_file_path_mobility_coefficient_no_lazio_only}")

## Evoluzione del Coefficiente di Mobilità Regionale per Anno

Per capire come il 'Coefficiente di Mobilità Regionale' si è evoluto nel tempo, calcoliamo la media di questo coefficiente per ciascun anno e visualizziamo la tendenza.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Unisci il DataFrame region_variety_counts con df_geo per ottenere l'anno
# Assicurati che 'IMDb_ID' in df_geo sia dello stesso tipo di region_variety_counts
# E che df_geo sia il DataFrame originale con le colonne 'IMDb_ID' e 'anno'

# Pulizia preliminare della colonna 'anno' in df_geo, se non già fatta in modo persistente
df_geo_temp = df_geo[['IMDb_ID', 'anno']].copy()
df_geo_temp.dropna(subset=['anno'], inplace=True)
df_geo_temp['anno'] = pd.to_numeric(df_geo_temp['anno'], errors='coerce')
df_geo_temp.dropna(subset=['anno'], inplace=True)
df_geo_temp['anno'] = df_geo_temp['anno'].astype(int)

# Rimuovi i duplicati di IMDb_ID e anno per evitare conteggi multipli per film con più location nello stesso anno
df_geo_temp = df_geo_temp.drop_duplicates(subset=['IMDb_ID', 'anno'])

# Unisci i dati del coefficiente di mobilità con gli anni
mobility_with_years = pd.merge(
    region_variety_counts[['IMDb_ID', 'Mobility_Coefficient']],
    df_geo_temp[['IMDb_ID', 'anno']],
    on='IMDb_ID',
    how='inner'
)

# Calcola la media del Coefficiente di Mobilità per ogni anno
annual_mobility_average = mobility_with_years.groupby('anno')['Mobility_Coefficient'].mean().reset_index()

# Filtra per anni con un numero sufficiente di film, se necessario, per avere dati significativi
# Qui non applichiamo un filtro, ma potresti volerlo fare per anni con pochi film.

# Visualizza l'evoluzione della media del Coefficiente di Mobilità nel tempo
plt.figure(figsize=(14, 7))
sns.lineplot(x='anno', y='Mobility_Coefficient', data=annual_mobility_average)
plt.title('Evoluzione del Coefficiente di Mobilità Regionale (Media Annuale)')
plt.xlabel('Anno')
plt.ylabel('Coefficiente di Mobilità Medio')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Save annual_mobility_average to CSV
csv_file_path_annual_mobility_average = '/content/annual_mobility_average.csv'
annual_mobility_average.to_csv(csv_file_path_annual_mobility_average, index=False)
print(f"File CSV salvato in: {csv_file_path_annual_mobility_average}")

## Evoluzione del Coefficiente di Mobilità Regionale per Quinquennio (1944-2025)

Per analizzare l'evoluzione in blocchi di 5 anni, calcoliamo la media del coefficiente di mobilità per ogni quinquennio.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# df_geo_temp e mobility_with_years sono già stati preparati nella cella precedente.
# Procediamo con il calcolo del quinquennio.

# Filtra per l'intervallo di anni specificato (1944-2025)
mobility_filtered_quinquennial = mobility_with_years[
    (mobility_with_years['anno'] >= 1944) &
    (mobility_with_years['anno'] <= 2025)
].copy()

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
mobility_filtered_quinquennial['quinquennial'] = (mobility_filtered_quinquennial['anno'] // 5) * 5

# Calcola la media del Coefficiente di Mobilità per ogni quinquennio
quinquennial_mobility_average = mobility_filtered_quinquennial.groupby('quinquennial')['Mobility_Coefficient'].mean().reset_index()

# Visualizza l'evoluzione della media del Coefficiente di Mobilità per quinquennio
plt.figure(figsize=(14, 7))
sns.lineplot(x='quinquennial', y='Mobility_Coefficient', data=quinquennial_mobility_average)
plt.title('Evoluzione del Coefficiente di Mobilità Regionale (Media per Quinquennio)')
plt.xlabel('Quinquennio')
plt.ylabel('Coefficiente di Mobilità Medio')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(quinquennial_mobility_average['quinquennial'], rotation=45)
plt.tight_layout()
plt.show()

# Save quinquennial_mobility_average to CSV
csv_file_path_quinquennial_mobility_average = '/content/quinquennial_mobility_average.csv'
quinquennial_mobility_average.to_csv(csv_file_path_quinquennial_mobility_average, index=False)
print(f"File CSV salvato in: {csv_file_path_quinquennial_mobility_average}")

Distribuzione per classi: non solo media, ma quota di film con mobilità 0, 0.25, 0.5, 0.75, 1 per anno/decennio

## Distribuzione Annuale del Coefficiente di Mobilità per Classi

Per comprendere meglio l'evoluzione della mobilità regionale, analizzeremo la quota di film per ogni specifica classe del 'Coefficiente di Mobilità' (0, 0.25, 0.5, 0.75, 1) per ciascun anno.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assicurati che mobility_with_years sia stato calcolato (dovrebbe esserlo dalle celle precedenti)
# mobility_with_years contiene 'IMDb_ID', 'Mobility_Coefficient', 'anno'

# Filtra i dati per l'intervallo di anni desiderato (es. 1944-2025 o l'intero range disponibile)
mobility_data_filtered = mobility_with_years[
    (mobility_with_years['anno'] >= 1944) &
    (mobility_with_years['anno'] <= 2025)
].copy()

# Calcola i conteggi di ogni coefficiente di mobilità per anno
annual_mobility_counts = mobility_data_filtered.groupby(['anno', 'Mobility_Coefficient']).size().unstack(fill_value=0)

# Calcola il totale dei film per ogni anno
annual_total_films = mobility_data_filtered.groupby('anno').size()

# Calcola le proporzioni (frequenza relativa) di ogni coefficiente per anno
annual_mobility_proportions = annual_mobility_counts.div(annual_total_films, axis=0)

# Rimuovi l'indice per preparare i dati per il melting e la visualizzazione
annual_mobility_proportions = annual_mobility_proportions.reset_index()

# Melt il DataFrame per un formato più adatto a seaborn
df_mobility_proportions_melted = annual_mobility_proportions.melt(
    id_vars=['anno'],
    var_name='Mobility_Coefficient_Class',
    value_name='Proportion'
)

# Ordina per anno per una visualizzazione corretta delle serie temporali
df_mobility_proportions_melted.sort_values(by='anno', inplace=True)

plt.figure(figsize=(16, 9))
sns.lineplot(
    x='anno',
    y='Proportion',
    hue='Mobility_Coefficient_Class',
    data=df_mobility_proportions_melted,
    marker='o', # Aggiungi marker per distinguere i punti dati
    palette='viridis' # Scegli una palette di colori
)

plt.title('Distribuzione Annuale della Quota di Film per Classe di Mobilità Regionale (1944-2025)')
plt.xlabel('Anno')
plt.ylabel('Quota di Film')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.legend(title='Coefficiente di Mobilità', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Questo grafico mostra come la proporzione di film in ciascuna classe di mobilità regionale è cambiata nel corso degli anni. Ogni linea rappresenta una specifica classe del coefficiente di mobilità (0, 0.25, 0.5, 0.75, 1), permettendoti di identificare tendenze e periodi in cui determinate classi di mobilità sono state più o meno prevalenti.

## Distribuzione per Quinquennio del Coefficiente di Mobilità per Classi

Analizziamo ora la quota di film per ogni specifica classe del 'Coefficiente di Mobilità' (0, 0.25, 0.5, 0.75, 1) per ciascun quinquennio, dal 1944 al 2025.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# mobility_with_years contains 'IMDb_ID', 'Mobility_Coefficient', 'anno'

# Filtra i dati per l'intervallo di anni desiderato (1944-2025)
mobility_data_filtered = mobility_with_years[
    (mobility_with_years['anno'] >= 1944) &
    (mobility_with_years['anno'] <= 2025)
].copy()

# Crea la colonna 'quinquennial' per raggruppare gli anni in blocchi di 5
mobility_data_filtered['quinquennial'] = (mobility_data_filtered['anno'] // 5) * 5

# Calcola i conteggi di ogni coefficiente di mobilità per quinquennio
quinquennial_mobility_counts = mobility_data_filtered.groupby(['quinquennial', 'Mobility_Coefficient']).size().unstack(fill_value=0)

# Calcola il totale dei film per ogni quinquennio
quinquennial_total_films = mobility_data_filtered.groupby('quinquennial').size()

# Calcola le proporzioni (frequenza relativa) di ogni coefficiente per quinquennio
quinquennial_mobility_proportions = quinquennial_mobility_counts.div(quinquennial_total_films, axis=0)

# Rimuovi l'indice per preparare i dati per il melting e la visualizzazione
quinquennial_mobility_proportions = quinquennial_mobility_proportions.reset_index()

# Melt il DataFrame per un formato più adatto a seaborn
df_quinquennial_mobility_proportions_melted = quinquennial_mobility_proportions.melt(
    id_vars=['quinquennial'],
    var_name='Mobility_Coefficient_Class',
    value_name='Proportion'
)

# Ordina per quinquennio per una visualizzazione corretta delle serie temporali
df_quinquennial_mobility_proportions_melted.sort_values(by='quinquennial', inplace=True)

plt.figure(figsize=(16, 9))
sns.lineplot(
    x='quinquennial',
    y='Proportion',
    hue='Mobility_Coefficient_Class',
    data=df_quinquennial_mobility_proportions_melted,
    marker='o',
    palette='viridis'
)

plt.title('Distribuzione Quinquennale della Quota di Film per Classe di Mobilità Regionale (1944-2025)')
plt.xlabel('Quinquennio')
plt.ylabel('Quota di Film')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(quinquennial_mobility_proportions['quinquennial'], rotation=45)
plt.legend(title='Coefficiente di Mobilità', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Distribuzione per Quinquennio andamento di film mono-regionali e pluri-regionali

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure df_quinquennial_mobility_proportions_melted is available from previous cells
# It contains 'quinquennial', 'Mobility_Coefficient_Class', 'Proportion'

# Define the categories for mono-regional and multi-regional films
# Mono-regional films have Mobility_Coefficient_Class == 0
# Multi-regional films have Mobility_Coefficient_Class > 0 (0.25, 0.5, 0.75, 1.0)

# Filter for mono-regional films
mono_regional_films = df_quinquennial_mobility_proportions_melted[
    df_quinquennial_mobility_proportions_melted['Mobility_Coefficient_Class'] == 0.0
].copy()
mono_regional_films['Type'] = 'Mono-regional films'

# Filter for multi-regional films and sum their proportions
multi_regional_films_df = df_quinquennial_mobility_proportions_melted[
    df_quinquennial_mobility_proportions_melted['Mobility_Coefficient_Class'] > 0.0
].copy()

# Group by quinquennial and sum the proportions for multi-regional films
multi_regional_films_grouped = multi_regional_films_df.groupby('quinquennial')['Proportion'].sum().reset_index()
multi_regional_films_grouped['Type'] = 'Multi-regional films'

# Combine the two dataframes
trend_analysis_df = pd.concat([mono_regional_films[['quinquennial', 'Proportion', 'Type']],
                               multi_regional_films_grouped[['quinquennial', 'Proportion', 'Type']]])

# Sort by quinquennial for proper line plot order
trend_analysis_df.sort_values(by='quinquennial', inplace=True)

# Save to CSV
csv_file_path_quinquennial_mobility_trend = '/content/quinquennial_mobility_trend.csv'
trend_analysis_df.to_csv(csv_file_path_quinquennial_mobility_trend, index=False)
print(f"File CSV salvato in: {csv_file_path_quinquennial_mobility_trend}")

plt.figure(figsize=(16, 9))
sns.lineplot(
    x='quinquennial',
    y='Proportion',
    hue='Type',
    data=trend_analysis_df,
    marker='o',
    palette={'Mono-regional films': 'blue', 'Multi-regional films': 'red'}
)

plt.title('Andamento Quinquennale di Film Mono-regionali e Pluriregionali (1944-2025)')
plt.xlabel('Quinquennio')
plt.ylabel('Quota di Film')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(trend_analysis_df['quinquennial'].unique(), rotation=45)
plt.legend(title='Tipo di Film', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Transform the DataFrame as requested by the user
pivoted_quinquennial_mobility_trend_df = trend_analysis_df.pivot_table(
    index='Type',
    columns='quinquennial',
    values='Proportion'
).fillna(0)

# Save the pivoted DataFrame to a new CSV file
csv_file_path_pivoted_quinquennial_mobility_trend = '/content/quinquennial_mobility_trend_pivoted.csv'
pivoted_quinquennial_mobility_trend_df.to_csv(csv_file_path_pivoted_quinquennial_mobility_trend, index=True)
print(f"File CSV con dati pivot salvato in: {csv_file_path_pivoted_quinquennial_mobility_trend}")

LO STESSO MA ANNUALE

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure df_mobility_proportions_melted is available from previous cells
# It contains 'anno', 'Mobility_Coefficient_Class', 'Proportion'

# Define the categories for mono-regional and multi-regional films
# Mono-regional films have Mobility_Coefficient_Class == 0
# Multi-regional films have Mobility_Coefficient_Class > 0 (0.25, 0.5, 0.75, 1.0)

# Filter for mono-regional films
mono_regional_films_annual = df_mobility_proportions_melted[
    df_mobility_proportions_melted['Mobility_Coefficient_Class'] == 0.0
].copy()
mono_regional_films_annual['Type'] = 'Film mono-regionali'

# Filter for multi-regional films and sum their proportions
multi_regional_films_annual_df = df_mobility_proportions_melted[
    df_mobility_proportions_melted['Mobility_Coefficient_Class'] > 0.0
].copy()

# Group by anno and sum the proportions for multi-regional films
multi_regional_films_annual_grouped = multi_regional_films_annual_df.groupby('anno')['Proportion'].sum().reset_index()
multi_regional_films_annual_grouped['Type'] = 'Film pluriregionali'

# Combine the two dataframes
trend_analysis_annual_df = pd.concat([mono_regional_films_annual[['anno', 'Proportion', 'Type']],
                                     multi_regional_films_annual_grouped[['anno', 'Proportion', 'Type']]])

# Sort by anno for proper line plot order
trend_analysis_annual_df.sort_values(by='anno', inplace=True)

# Save to CSV
csv_file_path_annual_mobility_trend = '/content/annual_mobility_trend.csv'
trend_analysis_annual_df.to_csv(csv_file_path_annual_mobility_trend, index=False)
print(f"File CSV salvato in: {csv_file_path_annual_mobility_trend}")

plt.figure(figsize=(16, 9))
sns.lineplot(
    x='anno',
    y='Proportion',
    hue='Type',
    data=trend_analysis_annual_df,
    marker='o',
    palette={'Film mono-regionali': 'blue', 'Film pluriregionali': 'red'}
)

plt.title('Andamento Annuale di Film Mono-regionali e Pluriregionali (1944-2025)')
plt.xlabel('Anno')
plt.ylabel('Quota di Film')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.legend(title='Tipo di Film', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Transform the DataFrame as requested by the user
pivoted_annual_mobility_trend_df = trend_analysis_annual_df.pivot_table(
    index='Type',
    columns='anno',
    values='Proportion'
).fillna(0)

# Save the pivoted DataFrame to a new CSV file
csv_file_path_pivoted_annual_mobility_trend = '/content/annual_mobility_trend_pivoted.csv'
pivoted_annual_mobility_trend_df.to_csv(csv_file_path_pivoted_annual_mobility_trend, index=True)
print(f"File CSV con dati pivot salvato in: {csv_file_path_pivoted_annual_mobility_trend}")

## Coefficiente di Mobilità Medio per Regione

Calcoliamo ora il coefficiente di mobilità medio per ciascuna regione per capire quali regioni tendono ad avere film girati in un numero maggiore di località diverse.

In [ ]:
# Merge mobility_with_years (which contains IMDb_ID and Mobility_Coefficient) with df_geo to get the region.
# We need to make sure 'IMDb_ID' in df_geo is of the same type as in mobility_with_years.

df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)
mobility_with_years['IMDb_ID'] = mobility_with_years['IMDb_ID'].astype(str)

mobility_by_region = pd.merge(
    mobility_with_years[['IMDb_ID', 'Mobility_Coefficient']],
    df_geo[['IMDb_ID', 'region']],
    on='IMDb_ID',
    how='inner'
)

# Drop rows where 'region' is missing (though it should be handled in previous steps, good for robustness)
mobility_by_region.dropna(subset=['region'], inplace=True)

# For films shot in multiple locations within the same region, or in multiple regions,
# an IMDb_ID might appear multiple times with the same region.
# To calculate the regional average correctly, we first need to ensure unique IMDb_ID-region pairs
# for a film's contribution to a region, then consider the unique films per region for averaging.
# However, the Mobility_Coefficient is already film-specific (per IMDb_ID).
# So, if a film is associated with multiple regions, its coefficient contributes to each of those regions' averages.

# Group by region and calculate the mean Mobility_Coefficient
average_mobility_per_region = mobility_by_region.groupby('region')['Mobility_Coefficient'].mean().reset_index()

# Sort for better readability
average_mobility_per_region = average_mobility_per_region.sort_values(by='Mobility_Coefficient', ascending=False)

display(average_mobility_per_region.head(10))

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Mobility_Coefficient',
    y='region',
    data=average_mobility_per_region,
    palette='viridis'
)
plt.title('Coefficiente di Mobilità Medio per Regione')
plt.xlabel('Coefficiente di Mobilità Medio')
plt.ylabel('Regione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
csv_file_path_avg_mobility_region = '/content/average_mobility_per_region.csv'
average_mobility_per_region.to_csv(csv_file_path_avg_mobility_region, index=False)
print(f"File CSV salvato in: {csv_file_path_avg_mobility_region}")

Singole regioni in relazione alla media nazionale

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure IMDb_ID columns are of string type for consistent merging
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)
mobility_with_years['IMDb_ID'] = mobility_with_years['IMDb_ID'].astype(str)

# 1. Merge mobility_with_years (IMDb_ID, Mobility_Coefficient, anno) with df_geo (IMDb_ID, region)
# This associates each film's mobility coefficient and year with all regions it touched.
film_mobility_regional_quinquennial = pd.merge(
    mobility_with_years,
    df_geo[['IMDb_ID', 'region']],
    on='IMDb_ID',
    how='inner'
)

# Drop rows where 'region' is missing and clean region names
film_mobility_regional_quinquennial.dropna(subset=['region'], inplace=True)
film_mobility_regional_quinquennial['region'] = film_mobility_regional_quinquennial['region'].astype(str).str.strip()

# Filter for the relevant year range (e.g., 1944-2025)
film_mobility_regional_quinquennial = film_mobility_regional_quinquennial[
    (film_mobility_regional_quinquennial['anno'] >= 1944) &
    (film_mobility_regional_quinquennial['anno'] <= 2025)
].copy()

# Create the 'quinquennial' column for grouping
film_mobility_regional_quinquennial['quinquennial'] = (film_mobility_regional_quinquennial['anno'] // 5) * 5

# 2. Calculate the mean Mobility_Coefficient for each region per quinquennium
regional_quinquennial_mobility_average = film_mobility_regional_quinquennial.groupby(['quinquennial', 'region'])['Mobility_Coefficient'].mean().reset_index()

# 3. Prepare the national quinquennial average data (already calculated as quinquennial_mobility_average)
national_quinquennial_mobility_average = quinquennial_mobility_average.copy()
national_quinquennial_mobility_average['region'] = 'National Average' # Add a 'region' column for plotting

# Filter national average for the same quinquennial range (already done in quinquennial_mobility_average calculation)
# No further filtering needed here for quinquennial_mobility_average

# 4. Prepare data for faceted plotting: each region's data combined with the national average

# Get all unique regions (excluding the 'National Average' placeholder)
unique_regions = regional_quinquennial_mobility_average['region'].unique()

# List to store dataframes for faceting
dfs_for_faceting = []

for region_name in unique_regions:
    # Get data for the current region
    current_region_data = regional_quinquennial_mobility_average[regional_quinquennial_mobility_average['region'] == region_name].copy()

    # Combine current region data with national average data
    # The 'region' column will distinguish between the region's line and the national average line within a facet
    # The 'facet_region' column will be used by FacetGrid to create separate plots for each region

    # Data for the specific region
    current_region_data['facet_region'] = region_name
    dfs_for_faceting.append(current_region_data)

    # National average data, assigned to this specific facet
    national_avg_for_facet = national_quinquennial_mobility_average.copy()
    national_avg_for_facet['facet_region'] = region_name
    dfs_for_faceting.append(national_avg_for_facet)

# Concatenate all prepared dataframes
combined_faceted_data = pd.concat(dfs_for_faceting, ignore_index=True)

# Determine max y-limit for consistent scaling across all facets
max_mobility_coefficient = combined_faceted_data['Mobility_Coefficient'].max()

# 5. Plotting with FacetGrid
g = sns.FacetGrid(
    combined_faceted_data,
    col='facet_region',
    col_wrap=4, # Adjust as needed for desired layout
    height=3.5, # Height of each facet
    aspect=1.2, # Aspect ratio of each facet
    sharey=True # Essential for comparing across regions
)

# Map the lineplot function to each facet
def draw_line_plot(data, **kwargs):
    # Plot the regional line
    sns.lineplot(
        x='quinquennial',
        y='Mobility_Coefficient',
        hue='region',
        data=data[data['region'] != 'National Average'],
        marker='o',
        linewidth=1.5,
        alpha=0.8,
        errorbar=None,
        ax=plt.gca(),
        palette=['blue'] # Color for the region's line
    )
    # Plot the national average line
    sns.lineplot(
        x='quinquennial',
        y='Mobility_Coefficient',
        data=data[data['region'] == 'National Average'],
        color='black',
        linestyle='--',
        marker='x',
        linewidth=1,
        alpha=0.7,
        errorbar=None,
        label='National Average',
        ax=plt.gca()
    )

    # Add legend to each subplot
    plt.legend(title='Legenda', labels=[data['facet_region'].iloc[0], 'National Average'], loc='upper left')

g.map_dataframe(draw_line_plot)

# Set common labels and titles
g.set_axis_labels('Quinquennio', 'Coefficiente di Mobilità Medio')
g.set_titles(col_template='{col_name}') # Changed from '{col}' to '{col_name}'
g.tight_layout() # Adjust layout to prevent labels from being cut off
plt.suptitle('Evoluzione Quinquennale del Coefficiente di Mobilità Medio (Regionale vs. Nazionale)',
             y=1.02, # Adjust vertical position of suptitle
             fontsize=16)
plt.show()

Calcolare il valore di mobilità medio per regione nella finestra 2000-2025

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Filter the film_mobility_regional_quinquennial DataFrame for the 2000-2025 period
mobility_2000_2025 = film_mobility_regional_quinquennial[
    (film_mobility_regional_quinquennial['anno'] >= 2000) &
    (film_mobility_regional_quinquennial['anno'] <= 2025)
].copy()

# Calculate the mean Mobility_Coefficient for each region in the 2000-2025 period
average_mobility_2000_2025_per_region = mobility_2000_2025.groupby('region')['Mobility_Coefficient'].mean().reset_index()

# Sort the results for better readability
average_mobility_2000_2025_per_region = average_mobility_2000_2025_per_region.sort_values(by='Mobility_Coefficient', ascending=False)

display(average_mobility_2000_2025_per_region.head(10))

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Mobility_Coefficient',
    y='region',
    data=average_mobility_2000_2025_per_region,
    palette='viridis'
)
plt.title('Average Regional Mobility Coefficient (2000-2025)')
plt.xlabel('Average Mobility Coefficient')
plt.ylabel('Region')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

The table above shows the average mobility coefficient for each region during the 2000-2025 period, ordered from highest to lowest. This helps identify which regions tend to have films shot in a greater variety of locations within this specific timeframe.

In [ ]:
csv_file_path_mobility_2000_2025_per_region = '/content/average_mobility_2000_2025_per_region.csv'
average_mobility_2000_2025_per_region.to_csv(csv_file_path_mobility_2000_2025_per_region, index=False)
print(f"File CSV salvato in: {csv_file_path_mobility_2000_2025_per_region}")

CALCOLO UN INDICE DI AUTONOMIA REGIONALE

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure IMDb_ID in df_geo is of string type for consistent merging
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)

# Drop rows where 'region' is missing and clean region names
region_data_for_autonomy = df_geo[['IMDb_ID', 'region']].copy()
region_data_for_autonomy.dropna(subset=['region'], inplace=True)
region_data_for_autonomy['region'] = region_data_for_autonomy['region'].astype(str).str.strip()

# 1. Total number of unique films each region appears in
# This counts how many distinct IMDb_ID are associated with each region, regardless of how many locations within that region.
total_films_per_region = region_data_for_autonomy.groupby('region')['IMDb_ID'].nunique().reset_index()
total_films_per_region.rename(columns={'IMDb_ID': 'Total_Films_Appears_In'}, inplace=True)

# 2. Identify mono-regional films
# A film is mono-regional if its 'Region_Variety' (number of unique regions for that film) is 1.
mono_regional_films = region_variety_counts[region_variety_counts['Region_Variety'] == 1]

# 3. Get the region for each mono-regional film
# We merge mono_regional_films with region_data_for_autonomy to associate each IMDb_ID with its single region.
mono_regional_films_with_region = pd.merge(
    mono_regional_films[['IMDb_ID']],
    region_data_for_autonomy[['IMDb_ID', 'region']].drop_duplicates(), # Get unique IMDb_ID-region pairs
    on='IMDb_ID',
    how='inner'
)

# 4. Count mono-regional films per region
mono_regional_films_count_per_region = mono_regional_films_with_region.groupby('region')['IMDb_ID'].nunique().reset_index()
mono_regional_films_count_per_region.rename(columns={'IMDb_ID': 'Mono_Regional_Film_Count'}, inplace=True)

# 5. Merge all data and calculate the Autonomy Index
autonomy_index_df = pd.merge(
    total_films_per_region,
    mono_regional_films_count_per_region,
    on='region',
    how='left'
)

# Fill NaN values (regions with no mono-regional films) with 0
autonomy_index_df['Mono_Regional_Film_Count'] = autonomy_index_df['Mono_Regional_Film_Count'].fillna(0).astype(int)

# Calculate the Regional Autonomy Index
autonomy_index_df['Regional_Autonomy_Index'] = (
    autonomy_index_df['Mono_Regional_Film_Count'] /
    autonomy_index_df['Total_Films_Appears_In']
)

# Sort the results by Regional Autonomy Index in descending order
autonomy_index_df = autonomy_index_df.sort_values(by='Regional_Autonomy_Index', ascending=False).reset_index(drop=True)

display(autonomy_index_df)

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Regional_Autonomy_Index',
    y='region',
    data=autonomy_index_df,
    palette='viridis'
)
plt.title('Indice di Autonomia Regionale (Proporzione di Film Mono-Regionali)')
plt.xlabel('Indice di Autonomia Regionale')
plt.ylabel('Regione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

La tabella sopra mostra l'indice di autonomia regionale calcolato per ogni regione. Le colonne sono:

*   **region**: Il nome della regione.
*   **Total_Films_Appears_In**: Il numero totale di film unici in cui la regione compare (anche se insieme ad altre regioni).
*   **Mono_Regional_Film_Count**: Il numero di film in cui quella regione è l'unica regione di ambientazione.
*   **Regional_Autonomy_Index**: L'indice di autonomia regionale, calcolato come la proporzione di film mono-regionali rispetto al totale dei film in cui la regione appare. I valori più alti indicano una maggiore 'autonomia' cinematografica della regione.

In [ ]:
# Salva il DataFrame come file CSV
csv_file_path_autonomy_index = '/content/regional_autonomy_index.csv'
autonomy_index_df.to_csv(csv_file_path_autonomy_index, index=False)
print(f"File CSV salvato in: {csv_file_path_autonomy_index}")

Versione contemporanea (2000-2025)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure IMDb_ID in df_geo is of string type for consistent merging
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)

# Filter df_geo for the period 2000-2025
region_data_2000_2025 = df_geo[
    (df_geo['anno'] >= 2000) &
    (df_geo['anno'] <= 2025)
].copy()

# Drop rows where 'region' is missing and clean region names
region_data_for_autonomy_2000_2025 = region_data_2000_2025[['IMDb_ID', 'region']].copy()
region_data_for_autonomy_2000_2025.dropna(subset=['region'], inplace=True)
region_data_for_autonomy_2000_2025['region'] = region_data_for_autonomy_2000_2025['region'].astype(str).str.strip()

# 1. Total number of unique films each region appears in (2000-2025)
total_films_per_region_2000_2025 = region_data_for_autonomy_2000_2025.groupby('region')['IMDb_ID'].nunique().reset_index()
total_films_per_region_2000_2025.rename(columns={'IMDb_ID': 'Total_Films_Appears_In'}, inplace=True)

# 2. Calculate Region_Variety per film for the 2000-2025 period
# This gives how many unique regions each film appears in for the 2000-2025 period
film_region_variety_2000_2025 = region_data_for_autonomy_2000_2025.groupby('IMDb_ID')['region'].nunique().reset_index()
film_region_variety_2000_2025.rename(columns={'region': 'Region_Variety'}, inplace=True)

# 3. Identify mono-regional films (Region_Variety == 1) for the 2000-2025 period
mono_regional_films_2000_2025 = film_region_variety_2000_2025[film_region_variety_2000_2025['Region_Variety'] == 1]

# 4. Get the region for each mono-regional film (2000-2025)
mono_regional_films_with_region_2000_2025 = pd.merge(
    mono_regional_films_2000_2025[['IMDb_ID']],
    region_data_for_autonomy_2000_2025[['IMDb_ID', 'region']].drop_duplicates(),
    on='IMDb_ID',
    how='inner'
)

# 5. Count mono-regional films per region (2000-2025)
mono_regional_films_count_per_region_2000_2025 = mono_regional_films_with_region_2000_2025.groupby('region')['IMDb_ID'].nunique().reset_index()
mono_regional_films_count_per_region_2000_2025.rename(columns={'IMDb_ID': 'Mono_Regional_Film_Count'}, inplace=True)

# 6. Merge all data and calculate the Autonomy Index for 2000-2025
autonomy_index_2000_2025_df = pd.merge(
    total_films_per_region_2000_2025,
    mono_regional_films_count_per_region_2000_2025,
    on='region',
    how='left'
)

# Fill NaN values (regions with no mono-regional films) with 0
autonomy_index_2000_2025_df['Mono_Regional_Film_Count'] = autonomy_index_2000_2025_df['Mono_Regional_Film_Count'].fillna(0).astype(int)

# Calculate the Regional Autonomy Index
autonomy_index_2000_2025_df['Regional_Autonomy_Index'] = (
    autonomy_index_2000_2025_df['Mono_Regional_Film_Count'] /
    autonomy_index_2000_2025_df['Total_Films_Appears_In']
)

# Sort the results by Regional Autonomy Index in descending order
autonomy_index_2000_2025_df = autonomy_index_2000_2025_df.sort_values(by='Regional_Autonomy_Index', ascending=False).reset_index(drop=True)

display(autonomy_index_2000_2025_df)

# Visualize the results
plt.figure(figsize=(12, 8))
sns.barplot(
    x='Regional_Autonomy_Index',
    y='region',
    data=autonomy_index_2000_2025_df,
    palette='viridis'
)
plt.title('Indice di Autonomia Regionale (2000-2025)')
plt.xlabel('Indice di Autonomia Regionale')
plt.ylabel('Regione')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Save the DataFrame as a CSV file
csv_file_path_autonomy_index_2000_2025 = '/content/regional_autonomy_index_2000_2025.csv'
autonomy_index_2000_2025_df.to_csv(csv_file_path_autonomy_index_2000_2025, index=False)
print(f"File CSV salvato in: {csv_file_path_autonomy_index_2000_2025}")

Confronto con media nazionale per quinquennio

### Calcolo dell'Indice di Autonomia Regionale per Quinquennio

Per analizzare l'evoluzione dell'indice di autonomia regionale nel tempo, calcoleremo l'indice per ogni regione per ogni blocco di cinque anni (quinquennio). Questo verrà poi confrontato con l'andamento medio nazionale per lo stesso periodo.

In [ ]:
# Ensure IMDb_ID is string type for consistent merges
df_geo['IMDb_ID'] = df_geo['IMDb_ID'].astype(str)

# Prepare a base DataFrame with IMDb_ID, region, and anno, then add quinquennial
# Re-using region_quinquennial_analysis_df from a previous step, ensuring it's up-to-date
region_quinquennial_analysis_df = df_geo[['IMDb_ID', 'region', 'anno']].copy()
region_quinquennial_analysis_df.dropna(subset=['region', 'anno'], inplace=True)
region_quinquennial_analysis_df['anno'] = pd.to_numeric(region_quinquennial_analysis_df['anno'], errors='coerce')
region_quinquennial_analysis_df.dropna(subset=['anno'], inplace=True)
region_quinquennial_analysis_df['anno'] = region_quinquennial_analysis_df['anno'].astype(int)
region_quinquennial_analysis_df = region_quinquennial_analysis_df[
    (region_quinquennial_analysis_df['anno'] >= 1944) &
    (region_quinquennial_analysis_df['anno'] <= 2025)
]
region_quinquennial_analysis_df['quinquennial'] = (region_quinquennial_analysis_df['anno'] // 5) * 5

# 1. Calculate Region_Variety per film per quinquennium
# This gives how many unique regions each film appears in for each quinquennium
film_region_variety_quinquennial = region_quinquennial_analysis_df.groupby(['IMDb_ID', 'quinquennial'])['region'].nunique().reset_index()
film_region_variety_quinquennial.rename(columns={'region': 'Region_Variety'}, inplace=True)

# 2. Identify mono-regional films (Region_Variety == 1) for each quinquennium
mono_regional_films_quinquennial = film_region_variety_quinquennial[
    film_region_variety_quinquennial['Region_Variety'] == 1
]

# 3. Merge mono-regional films back with the original df to get the specific region for these films
# We need to ensure we only count each IMDb_ID-region pair once for mono-regional films
mono_regional_films_with_region_quinquennial = pd.merge(
    mono_regional_films_quinquennial[['IMDb_ID', 'quinquennial']],
    region_quinquennial_analysis_df[['IMDb_ID', 'quinquennial', 'region']].drop_duplicates(),
    on=['IMDb_ID', 'quinquennial'],
    how='inner'
)

# 4. Count mono-regional films per region per quinquennium
mono_regional_counts_per_quinquennial_region = mono_regional_films_with_region_quinquennial.groupby(['quinquennial', 'region'])['IMDb_ID'].nunique().reset_index()
mono_regional_counts_per_quinquennial_region.rename(columns={'IMDb_ID': 'Mono_Regional_Film_Count'}, inplace=True)

# 5. Total films per region per quinquennium
total_films_per_quinquennial_region = region_quinquennial_analysis_df.groupby(['quinquennial', 'region'])['IMDb_ID'].nunique().reset_index()
total_films_per_quinquennial_region.rename(columns={'IMDb_ID': 'Total_Films_Appears_In'}, inplace=True)

# 6. Merge to calculate Regional Autonomy Index per quinquennium
autonomy_index_quinquennial_regional = pd.merge(
    total_films_per_quinquennial_region,
    mono_regional_counts_per_quinquennial_region,
    on=['quinquennial', 'region'],
    how='left'
)

autonomy_index_quinquennial_regional['Mono_Regional_Film_Count'] = autonomy_index_quinquennial_regional['Mono_Regional_Film_Count'].fillna(0).astype(int)

autonomy_index_quinquennial_regional['Regional_Autonomy_Index'] = (
    autonomy_index_quinquennial_regional['Mono_Regional_Film_Count'] /
    autonomy_index_quinquennial_regional['Total_Films_Appears_In']
)

display(autonomy_index_quinquennial_regional.head())

### Calcolo dell'Indice di Autonomia Nazionale per Quinquennio

In [ ]:
# 7. Calculate National Average Autonomy Index per Quinquennium
national_total_films_per_quinquennial = total_films_per_quinquennial_region.groupby('quinquennial')['Total_Films_Appears_In'].sum().reset_index()
national_total_films_per_quinquennial.rename(columns={'Total_Films_Appears_In': 'National_Total_Films'}, inplace=True)

national_mono_regional_films_per_quinquennial = mono_regional_counts_per_quinquennial_region.groupby('quinquennial')['Mono_Regional_Film_Count'].sum().reset_index()
national_mono_regional_films_per_quinquennial.rename(columns={'Mono_Regional_Film_Count': 'National_Mono_Regional_Film_Count'}, inplace=True)

national_autonomy_quinquennial = pd.merge(
    national_total_films_per_quinquennial,
    national_mono_regional_films_per_quinquennial,
    on='quinquennial',
    how='left'
)
national_autonomy_quinquennial['National_Mono_Regional_Film_Count'] = national_autonomy_quinquennial['National_Mono_Regional_Film_Count'].fillna(0).astype(int)
national_autonomy_quinquennial['National_Autonomy_Index'] = (
    national_autonomy_quinquennial['National_Mono_Regional_Film_Count'] /
    national_autonomy_quinquennial['National_Total_Films']
)

display(national_autonomy_quinquennial.head())


### Visualizzazione dell'Evoluzione dell'Indice di Autonomia Regionale vs. Nazionale per Quinquennio

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare for plotting
# Get unique regions
unique_regions = autonomy_index_quinquennial_regional['region'].unique()

dfs_for_faceting_autonomy = []

# Loop through each region and create combined data for faceting
for region_name in unique_regions:
    # Data for the current region
    current_region_autonomy_data = autonomy_index_quinquennial_regional[
        autonomy_index_quinquennial_regional['region'] == region_name
    ].copy()

    # Combine with national average for the same quinquennia
    # Ensure national average has a 'region' column for distinguishing lines in the plot
    national_avg_for_facet_autonomy = national_autonomy_quinquennial[['quinquennial', 'National_Autonomy_Index']].copy()
    national_avg_for_facet_autonomy.rename(columns={'National_Autonomy_Index': 'Regional_Autonomy_Index'}, inplace=True)
    national_avg_for_facet_autonomy['region'] = 'National Average'

    # Concatenate the current region's data with the national average data
    # Create a 'plot_group' column to distinguish between the region's line and the national average line within the facet
    current_region_autonomy_data['plot_group'] = region_name
    national_avg_for_facet_autonomy['plot_group'] = region_name

    dfs_for_faceting_autonomy.append(current_region_autonomy_data)
    dfs_for_faceting_autonomy.append(national_avg_for_facet_autonomy)


combined_faceted_autonomy_data = pd.concat(dfs_for_faceting_autonomy, ignore_index=True)

# Determine max y-limit for consistent scaling across all facets
max_autonomy_index = combined_faceted_autonomy_data['Regional_Autonomy_Index'].max()

# Plotting with FacetGrid
g = sns.FacetGrid(
    combined_faceted_autonomy_data,
    col='plot_group',
    col_wrap=4, # Adjust as needed for desired layout
    height=3.5, # Height of each facet
    aspect=1.2, # Aspect ratio of each facet
    sharey=True # Essential for comparing across regions
)

# Map the lineplot function to each facet
def draw_line_plot_autonomy(data, **kwargs):
    # Plot the regional line
    sns.lineplot(
        x='quinquennial',
        y='Regional_Autonomy_Index',
        hue='region',
        data=data[data['region'] != 'National Average'],
        marker='o',
        linewidth=1.5,
        alpha=0.8,
        errorbar=None,
        ax=plt.gca(),
        palette=['blue'] # Color for the region's line
    )
    # Plot the national average line
    sns.lineplot(
        x='quinquennial',
        y='Regional_Autonomy_Index',
        data=data[data['region'] == 'National Average'],
        color='black',
        linestyle='--',
        marker='x',
        linewidth=1,
        alpha=0.7,
        errorbar=None,
        label='National Average',
        ax=plt.gca()
    )

    # Add legend to each subplot
    # Filter labels to only show the region name and National Average, preventing duplicates
    handles, labels = plt.gca().get_legend_handles_labels()
    filtered_labels = [label for label in labels if label != data['plot_group'].iloc[0]] # Remove the plot_group label if present
    plt.legend(handles, [data['plot_group'].iloc[0]] + [l for l in filtered_labels if l != data['plot_group'].iloc[0]], title='Legenda', loc='upper left')

    # Set y-axis limits to ensure consistency and readability
    plt.ylim(0, max_autonomy_index * 1.1) # 10% buffer above max value

g.map_dataframe(draw_line_plot_autonomy)

# Set common labels and titles
g.set_axis_labels('Quinquennio', 'Indice di Autonomia Regionale')
g.set_titles(col_template='{col_name}') # Changed from '{col}' to '{col_name}'
plt.suptitle('Evoluzione Quinquennale dell\'Indice di Autonomia Regionale (Regionale vs. Nazionale)',
             y=1.02, # Adjust vertical position of suptitle
             fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to prevent labels from being cut off and make space for suptitle
plt.show()

### Scarico il file CSV con l'Indice di Autonomia Regionale per Quinquennio

In [ ]:
csv_file_path_autonomy_quinquennial = '/content/regional_autonomy_index_quinquennial.csv'
autonomy_index_quinquennial_regional.to_csv(csv_file_path_autonomy_quinquennial, index=False)
print(f"File CSV salvato in: {csv_file_path_autonomy_quinquennial}")

TIPOLOGIA INTERPRETATIVA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assumiamo che `autonomy_index_2000_2025_df` e `region_relative_frequency_2000_2025_df` siano già calcolati e disponibili.
# Questi DataFrame dovrebbero contenere la colonna 'region' come chiave di unione.

# Unisci i due DataFrame sulla colonna 'region'
merged_quadrant_data = pd.merge(
    autonomy_index_2000_2025_df[['region', 'Regional_Autonomy_Index']],
    region_relative_frequency_2000_2025_df[['region', 'Relative_IMDb_Frequency_2000_2025']],
    on='region',
    how='inner'
)

# Rinominare le colonne per chiarezza nel plot, se necessario (già fatto per l'autonomia)
merged_quadrant_data.rename(columns={'Relative_IMDb_Frequency_2000_2025': 'Incidenza_Nazionale'},
                            inplace=True)

# Calcola le mediane dei due indicatori per definire i quadranti
median_incidenza = merged_quadrant_data['Incidenza_Nazionale'].median()
median_autonomia = merged_quadrant_data['Regional_Autonomy_Index'].median()

# Assegna a ogni regione una categoria basata sulle mediane
def assign_quadrant_category(row):
    if row['Incidenza_Nazionale'] >= median_incidenza and row['Regional_Autonomy_Index'] >= median_autonomia:
        return 'centrale' # alta incidenza, alta autonomia
    elif row['Incidenza_Nazionale'] < median_incidenza and row['Regional_Autonomy_Index'] >= median_autonomia:
        return 'periferica autonoma' # bassa incidenza, alta autonomia
    elif row['Incidenza_Nazionale'] >= median_incidenza and row['Regional_Autonomy_Index'] < median_autonomia:
        return 'di snodo' # alta incidenza, bassa autonomia
    else:
        return 'subordinata' # bassa incidenza, bassa autonomia

merged_quadrant_data['category'] = merged_quadrant_data.apply(assign_quadrant_category, axis=1)

# Imposta lo stile del plot
sns.set_style('whitegrid')

plt.figure(figsize=(14, 10))
sns.scatterplot(
    x='Incidenza_Nazionale',
    y='Regional_Autonomy_Index',
    data=merged_quadrant_data,
    hue='category', # Colora i punti per categoria
    s=200, # Dimensione dei punti
    alpha=0.8,
    legend='full' # Mostra la legenda completa per le categorie
)

# Aggiungi etichette per ogni regione
for line in range(0, merged_quadrant_data.shape[0]):
     plt.text(
         merged_quadrant_data.Incidenza_Nazionale[line] + 0.0005,
         merged_quadrant_data.Regional_Autonomy_Index[line],
         merged_quadrant_data.region[line],
         horizontalalignment='left',
         size='small',
         color='black',
         weight='semibold'
     )

# Aggiungi linee per la mediana dei due indicatori per definire i quadranti
plt.axvline(median_incidenza, color='gray', linestyle='--', linewidth=1.5, label=f'Mediana Incidenza: {median_incidenza:.4f}')
plt.axhline(median_autonomia, color='gray', linestyle='--', linewidth=1.5, label=f'Mediana Autonomia: {median_autonomia:.4f}')


plt.title('Regional Typology of Italian Film Locations', fontsize=16)
plt.xlabel('Relative weight', fontsize=12)
plt.ylabel('Regional Autonomy Index', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Categoriy')
plt.tight_layout()
plt.show()

# Salva il DataFrame con le categorie dei quadranti in un nuovo file CSV
csv_file_path_quadrant_categories = '/content/regional_quadrant_categories_2000_2025.csv'
merged_quadrant_data.to_csv(csv_file_path_quadrant_categories, index=False)
print(f"File CSV con categorie salvato in: {csv_file_path_quadrant_categories}")

## Tipologia a Quattro Quadranti: Analisi Completa (Dataset Completo)

Questa sezione replica l'analisi tipologica a quattro quadranti, ma utilizzando l'intero dataset disponibile, non solo il periodo 2000-2025. Verranno calcolate l'incidenza nazionale e l'indice di autonomia regionale per l'intero arco temporale coperto dai dati, e le regioni verranno categorizzate e visualizzate di conseguenza.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Calcolo della "National Incidence/Weight of films per region" per il dataset completo ---

# Creazione di un DataFrame combinato con le colonne rilevanti da df_geo
# (df_geo è disponibile dalle celle precedenti e contiene tutti gli anni)
region_full_analysis_df = df_geo[['IMDb_ID', 'region']].copy()

# Rimuovi le righe con dati mancanti per 'region'
region_full_analysis_df.dropna(subset=['region'], inplace=True)

# Standardizza il nome della regione rimuovendo spazi bianchi e convertendolo in stringa
region_full_analysis_df['region'] = region_full_analysis_df['region'].astype(str).str.strip()

# Calcola i pesi: conta gli IMDb_ID unici per regione per l'intero blocco
# La presenza della stessa regione più volte per un singolo IMDb_ID vale sempre uno.
unique_associations_full = region_full_analysis_df.drop_duplicates(subset=['IMDb_ID', 'region'])

# Raggruppiamo per regione e contiamo gli IMDb_ID unici
region_counts_full = unique_associations_full.groupby('region')['IMDb_ID'].nunique().reset_index()

# Rinomina la colonna per chiarezza
region_counts_full.rename(columns={'IMDb_ID': 'Number of Unique IMDb_IDs'}, inplace=True)

# Calcola il totale di IMDb_ID unici nazionale per l'intero blocco
total_imdb_ids_national_full = region_counts_full['Number of Unique IMDb_IDs'].sum()

# Calcola la frequenza relativa per regione per l'intero blocco
region_relative_frequency_full_data = region_counts_full.copy()
region_relative_frequency_full_data['Incidenza_Nazionale'] = (
    region_relative_frequency_full_data['Number of Unique IMDb_IDs'] /
    total_imdb_ids_national_full
)

print("Frequenza Relativa IMDb per Regione (Dataset Completo):")
display(region_relative_frequency_full_data.head())

# --- 2. Unione con l'Indice di Autonomia Regionale (già calcolato come autonomy_index_df) ---
# autonomy_index_df è già disponibile e contiene 'region' e 'Regional_Autonomy_Index' per il dataset completo

merged_quadrant_data_full = pd.merge(
    region_relative_frequency_full_data[['region', 'Incidenza_Nazionale']],
    autonomy_index_df[['region', 'Regional_Autonomy_Index']],
    on='region',
    how='inner'
)

# --- 3. Calcolo delle mediane per definire i quadranti ---
median_incidenza_full = merged_quadrant_data_full['Incidenza_Nazionale'].median()
median_autonomia_full = merged_quadrant_data_full['Regional_Autonomy_Index'].median()

# --- 4. Assegnazione delle categorie ai quadranti ---
def assign_quadrant_category_full(row):
    if row['Incidenza_Nazionale'] >= median_incidenza_full and row['Regional_Autonomy_Index'] >= median_autonomia_full:
        return 'centrale' # alta incidenza, alta autonomia
    elif row['Incidenza_Nazionale'] < median_incidenza_full and row['Regional_Autonomy_Index'] >= median_autonomia_full:
        return 'periferica autonoma' # bassa incidenza, alta autonomia
    elif row['Incidenza_Nazionale'] >= median_incidenza_full and row['Regional_Autonomy_Index'] < median_autonomia_full:
        return 'di snodo' # alta incidenza, bassa autonomia
    else:
        return 'subordinata' # bassa incidenza, bassa autonomia

merged_quadrant_data_full['category'] = merged_quadrant_data_full.apply(assign_quadrant_category_full, axis=1)

# --- 5. Visualizzazione del grafico a dispersione ---
sns.set_style('whitegrid')

plt.figure(figsize=(14, 10))
sns.scatterplot(
    x='Incidenza_Nazionale',
    y='Regional_Autonomy_Index',
    data=merged_quadrant_data_full,
    hue='category',
    s=200,
    alpha=0.8,
    legend='full'
)

# Aggiungi etichette per ogni regione
for line in range(0, merged_quadrant_data_full.shape[0]):
     plt.text(
         merged_quadrant_data_full.Incidenza_Nazionale[line] + 0.0005,
         merged_quadrant_data_full.Regional_Autonomy_Index[line],
         merged_quadrant_data_full.region[line],
         horizontalalignment='left',
         size='small',
         color='black',
         weight='semibold'
     )

# Aggiungi linee per la mediana dei due indicatori
plt.axvline(median_incidenza_full, color='gray', linestyle='--', linewidth=1.5, label=f'Mediana Incidenza: {median_incidenza_full:.4f}')
plt.axhline(median_autonomia_full, color='gray', linestyle='--', linewidth=1.5, label=f'Mediana Autonomia: {median_autonomia_full:.4f}')

plt.title('Tipologia a Quattro Quadranti: Autonomia Regionale vs. Incidenza Nazionale (Dataset Completo)', fontsize=16)
plt.xlabel('Incidenza Nazionale (Frequenza Relativa IMDb)', fontsize=12)
plt.ylabel('Indice di Autonomia Regionale', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title='Categoria Quadrante')
plt.tight_layout()
plt.show()


Scarico il file CSV con le categorie dei quadranti per il dataset completo.

In [ ]:
csv_file_path_quadrant_categories_full = '/content/regional_quadrant_categories_full_dataset.csv'
merged_quadrant_data_full.to_csv(csv_file_path_quadrant_categories_full, index=False)
print(f"File CSV salvato in: {csv_file_path_quadrant_categories_full}")